# Packages and Data Instantiation

In [1]:
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import ndtr
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MultiLabelBinarizer

def load_csv_dir(path: str = "../data") -> Dict[str, pd.DataFrame]:
    """Load all CSV files in a directory into a dict keyed by filename stem."""
    return {f.stem: pd.read_csv(f, engine="pyarrow") for f in Path(path).glob("*.csv")}

def load_excel_dir(path: str = "../data") -> Dict[str, pd.DataFrame]:
    """Load all Excel files in a directory into a dict keyed by filename stem."""
    return {
        f.stem: pd.read_excel(f, engine="openpyxl", index_col=0, parse_dates=True)
        for f in Path(path).glob("*.xlsx")
    }

l_dir = load_csv_dir
l_xl = load_excel_dir

In [2]:
data_map = load_csv_dir()
d_m = data_map

df_crs = data_map.get("crsp_daily_prices")
df_evt = data_map.get("capitaliq_key_developments")
df_ff = data_map.get("fama_french_5f_daily")
df_ibs = data_map.get("ibes_eps_summary")
df_main = data_map.get("integrated_feature_matrix")

In [3]:
class EDA:
    """Exploratory checks for a time-series feature matrix."""

    def __init__(self, frame: pd.DataFrame):
        if not isinstance(frame, pd.DataFrame):
            raise ValueError("`frame` must be a pandas DataFrame.")
        self.frame = frame
        self.columns = frame.columns

    def missingness(self) -> pd.DataFrame:
        """Return missing counts, percentages, and dtypes by column."""
        missing_n = self.frame.isna().sum()
        missing_pct = missing_n.div(len(self.frame)).mul(100)
        dtypes = self.frame.dtypes
        return (
            pd.DataFrame({"n_ms": missing_n, "p_ms": missing_pct, "d_ty": dtypes})
            .sort_values("p_ms", ascending=False)
        )

    def time_span(self, date_col: str) -> Dict[str, str]:
        """Summarize date coverage and repeated-date count."""
        dt = pd.to_datetime(self.frame[date_col], errors="coerce").dropna()
        if dt.empty:
            return {"strt": "NA", "end": "NA", "n_dys": "0", "gaps": "0"}
        return {
            "strt": str(dt.min().date()),
            "end": str(dt.max().date()),
            "n_dys": str(dt.nunique()),
            "gaps": str(len(dt) - dt.nunique()),
        }

    def target_stats(self, target_col: str) -> pd.DataFrame:
        """Return descriptive statistics for a target variable."""
        if target_col not in self.columns:
            return pd.DataFrame()
        return self.frame[[target_col]].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

    # Backward-compatible method names used in existing cells.
    def s_chk(self) -> pd.DataFrame:
        return self.missingness()

    def t_chk(self, d_c: str) -> Dict[str, str]:
        return self.time_span(d_c)

    def tgt_sts(self, t_c: str) -> pd.DataFrame:
        return self.target_stats(t_c)

In [4]:
eda = EDA(df_main)
eda.s_chk()

,n_ms,p_ms,d_ty
headline,808,83.213182,object
category,808,83.213182,object
eventtype,808,83.213182,object
keydeveventtypeid,808,83.213182,object
dlyret,73,7.518023,float64
dlyvol,72,7.415036,float64
mktrf,0,0.000000,float64
fpedats_fy1,0,0.000000,object
fpedats_fy2,0,0.000000,object
curcode_fy1,0,0.000000,object


In [5]:
eda.t_chk('dlycaldt')

{'strt': '2022-06-02', 'end': '2026-04-10', 'n_dys': '971', 'gaps': '0'}

In [6]:
eda.tgt_sts('dlyret')

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
dlyret,898.0,0.000071,0.019793,-0.146109,-0.052006,-0.031786,-0.009884,-0.000206,0.012205,0.029974,0.049173,0.087826


# Econometrics

In [7]:
import numpy as np
import pandas as pd
from dask import compute, delayed
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, bds
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox, het_white, het_breuschpagan
from statsmodels.tsa.ar_model import ar_select_order, AutoReg
from statsmodels.stats.stattools import durbin_watson, jarque_bera
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning

class TSDiagnostics:
    def __init__(self, y: np.ndarray):
        self.y = np.ascontiguousarray(np.asarray(y, dtype=np.float64).ravel())
        self.y = self.y[np.isfinite(self.y)]
        self.dy = np.diff(self.y)
        self.n = self.y.shape[0]
        self.dy_ok = self.dy.size > 0 and np.ptp(self.dy) > 0.0

    @staticmethod
    def _run_test(func, out_dict, key_map, *args, **kwargs):
        try:
            res = func(*args, **kwargs)
            if res is not None:
                for key, idx in key_map.items():
                    out_dict[key] = res[idx] if isinstance(res, tuple) else res
        except Exception:
            pass

    @delayed
    def _test_stationarity(self, max_lag: int) -> dict:
        out = {}
        self._run_test(adfuller, out, {'adf_stat': 0, 'adf_pval': 1}, self.y, maxlag=max_lag, autolag='AIC')
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", InterpolationWarning)
            self._run_test(kpss, out, {'kpss_stat': 0, 'kpss_pval': 1}, self.y, regression='c', nlags='auto')

        if self.dy_ok:
            self._run_test(adfuller, out, {'adf_d_stat': 0, 'adf_d_pval': 1}, self.dy, maxlag=max_lag, autolag='AIC')
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", InterpolationWarning)
                self._run_test(kpss, out, {'kpss_d_stat': 0, 'kpss_d_pval': 1}, self.dy, regression='c', nlags='auto')
        return out

    @delayed
    def _test_serial_corr(self, lags: int) -> dict:
        out = {}
        try:
            out['lb_pval'] = acorr_ljungbox(self.y, lags=[lags], return_df=False).iloc[0, 1]
        except Exception:
            pass

        if self.dy_ok:
            try:
                out['lb_d_pval'] = acorr_ljungbox(self.dy, lags=[lags], return_df=False).iloc[0, 1]
                out['lb_abs_d_pval'] = acorr_ljungbox(np.abs(self.dy), lags=[lags], return_df=False).iloc[0, 1]
                out['lb_d2_pval'] = acorr_ljungbox(self.dy**2, lags=[lags], return_df=False).iloc[0, 1]
            except Exception:
                pass
            self._run_test(het_arch, out, {'arch_lm_pval': 1}, self.dy, nlags=lags)
        return out

    @delayed
    def _test_residuals(self) -> dict:
        out = {}
        xc = np.ones((self.n, 1), dtype=np.float64)
        xh = np.column_stack((np.ones(self.n, dtype=np.float64), np.arange(self.n, dtype=np.float64)))

        try:
            ols = sm.OLS(self.y, xc).fit()
            r = np.asarray(ols.resid, dtype=np.float64)

            try:
                out["dw_stat"] = float(durbin_watson(r))
            except Exception:
                pass

            self._run_test(het_white, out, {"hetw_lm_p": 1}, r, xh)
            self._run_test(het_breuschpagan, out, {"hetbp_lm_p": 1}, r, xh, robust=True)
            self._run_test(jarque_bera, out, {"jb_p": 1, "jb_skew": 2, "jb_kurt": 3}, r)

        except Exception:
            pass

        if self.dy_ok:
            try:
                s, p = bds(self.dy, max_dim=2)
                out.update({"bds_stat": s[0] if isinstance(s, np.ndarray) else s,
                            "bds_p": p[0] if isinstance(p, np.ndarray) else p})
            except Exception:
                pass
        return out

    @delayed
    def _fit_ar(self, max_lag: int) -> dict:
        out = {}
        try:
            sel = ar_select_order(self.y, maxlag=max_lag, ic='bic', trend='c')
            lags = sel.ar_lags if (sel.ar_lags is not None and len(sel.ar_lags) > 0) else [1]
            mod = AutoReg(self.y, lags=lags, trend='c').fit()
            out.update({'ar_lags': len(lags), 'ar_bic': mod.bic})
        except Exception:
            pass

        if self.dy_ok:
             try:
                sel_d = ar_select_order(self.dy, maxlag=max_lag, ic='bic', trend='c')
                lags_d = sel_d.ar_lags if (sel_d.ar_lags is not None and len(sel_d.ar_lags) > 0) else [1]
                mod_d = AutoReg(self.dy, lags=lags_d, trend='c').fit()
                out.update({'ar_d_lags': len(lags_d), 'ar_d_bic': mod_d.bic})
             except Exception:
                pass
        return out

    @classmethod
    def profile_assets(cls, df: pd.DataFrame, target_col: str, asset_col: str, min_obs: int = 50) -> pd.DataFrame:
        if asset_col not in df.columns:
            df = df.copy()
            df[asset_col] = 'TARGET'

        tasks = {}
        for asset, group in df.groupby(asset_col, sort=False):
            series = group[target_col].to_numpy()
            if np.isfinite(series).sum() >= min_obs:
                engine = cls(series)
                dyn_lag = max(1, min(12, engine.n // 10))

                t1 = engine._test_stationarity(dyn_lag)
                t2 = engine._test_serial_corr(dyn_lag)
                t3 = engine._test_residuals()
                t4 = engine._fit_ar(dyn_lag)

                tasks[asset] = delayed(lambda *dicts: {k: v for d in dicts for k, v in d.items()})(t1, t2, t3, t4)

        results = compute(tasks)[0]
        return pd.DataFrame.from_dict(results, orient='index')

In [8]:
import yfinance as yf

YF_START = "2000-01-01"
YF_END = "2026-04-07"
YF_SYMBOL = "WDS.AX"

def fetch_yf_history(symbol: str, start: str = YF_START, end: str = YF_END) -> pd.DataFrame:
    """Download normalized daily price, return, and volume series from yfinance."""
    end_next = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    raw = yf.download(
        symbol,
        start=start,
        end=end_next,
        auto_adjust=True,
        progress=False,
        actions=False,
    )
    if raw.empty:
        raise ValueError(f"No rows returned for {symbol} in [{start}, {end}]")

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    close_col = "Adj Close" if "Adj Close" in raw.columns else "Close"
    out = pd.DataFrame(
        {
            "dlyprc": pd.to_numeric(raw[close_col], errors="coerce"),
            "dlyvol": pd.to_numeric(raw["Volume"], errors="coerce") if "Volume" in raw.columns else np.nan,
        },
        index=raw.index,
    ).dropna(subset=["dlyprc"])

    out["dlyret"] = out["dlyprc"].pct_change()
    out = out.reset_index().rename(columns={"Date": "dlycaldt"})
    out["dlycaldt"] = pd.to_datetime(out["dlycaldt"]).dt.normalize()
    out["ticker"] = symbol
    return out[["dlycaldt", "dlyprc", "dlyret", "dlyvol", "ticker"]]

def first_yf_history(symbols: list[str], start: str = YF_START, end: str = YF_END) -> pd.DataFrame:
    """Compatibility wrapper that runs the first symbol only."""
    if not symbols:
        raise ValueError("symbols must contain at least one ticker")
    return fetch_yf_history(symbol=symbols[0], start=start, end=end)

get_yf_prices_returns = fetch_yf_history
load_yf_prices_returns = first_yf_history

In [9]:
df_yf = fetch_yf_history(YF_SYMBOL, start=YF_START, end=YF_END)
print(
    f"Using yfinance ticker {YF_SYMBOL}. Rows: {len(df_yf)}. "
    f"Range: {df_yf['dlycaldt'].min().date()} to {df_yf['dlycaldt'].max().date()}"
)

df_econometrics = TSDiagnostics.profile_assets(
    df_yf,
    target_col="dlyret",
    asset_col="ticker",
)
print(df_econometrics)

df_econ_input = df_yf.copy()

Using yfinance ticker WDS.AX. Rows: 6680. Range: 2000-01-03 to 2026-04-07
         adf_stat  adf_pval  kpss_stat  kpss_pval  adf_d_stat  adf_d_pval  \
WDS.AX -34.613993       0.0   0.114478        0.1  -38.235044         0.0   

        kpss_d_stat  kpss_d_pval   lb_pval  lb_d_pval  ...   dw_stat  \
WDS.AX     0.089093          0.1  0.018806        0.0  ...  1.990443   

        hetw_lm_p  hetbp_lm_p  jb_p   jb_skew  jb_kurt  ar_lags        ar_bic  \
WDS.AX   0.281381    0.579739   0.0 -0.107758  8.86942        1 -34550.444508   

        ar_d_lags      ar_d_bic  
WDS.AX         12 -33910.614808  

[1 rows x 23 columns]


In [10]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Sequence

import numpy as np
import pandas as pd
from arch import arch_model
from sklearn.neighbors import NearestNeighbors


@dataclass(slots=True)
class GARCHFit:
    """Container for a fitted AR(1)-GJR-GARCH(1,1)-t specification.

    Attributes:
        mu: Constant of the AR(1) mean equation.
        phi: AR(1) coefficient.
        omega: GARCH intercept.
        alpha: ARCH coefficient on squared innovations.
        gamma: Asymmetry coefficient (leverage effect).
        beta: GARCH coefficient on lagged variance.
        nu: Student-t degrees of freedom.
        sigma0_sq: Current conditional variance (end of sample).
        mean0: Current conditional mean (end of sample, for AR drift seed).
        residuals: Standardized residuals z_t = eps_t / sigma_t.
        loglik: In-sample log-likelihood.
    """
    mu: float
    phi: float
    omega: float
    alpha: float
    gamma: float
    beta: float
    nu: float
    sigma0_sq: float
    mean0: float
    residuals: np.ndarray
    loglik: float


def fit_gjr_garch(returns: pd.Series) -> GARCHFit:
    """Fits AR(1)-GJR-GARCH(1,1) with Student-t innovations.

    The arch library expects returns in percent for numerical stability;
    we convert on entry and strip the 100x scale from the coefficients
    on exit so simulation operates on raw returns throughout.

    Args:
        returns: Daily simple returns as a pandas Series.

    Returns:
        Fitted GARCHFit container with the current state seed for simulation.
    """
    r = returns.dropna().astype(np.float64) * 100.0
    model = arch_model(r, mean="AR", lags=1, vol="GARCH", p=1, o=1, q=1, dist="studentst", rescale=False)
    fit = model.fit(disp="off", show_warning=False)

    params = fit.params
    cond_vol = fit.conditional_volatility.to_numpy()
    resid = fit.resid.to_numpy()

    sigma0_sq = float(cond_vol[-1] ** 2) / (100.0 ** 2)
    mean0 = float(r.iloc[-1]) / 100.0
    standardized = (resid / cond_vol)[np.isfinite(resid / cond_vol)]

    return GARCHFit(
        mu=float(params["Const"]) / 100.0,
        phi=float(params["dlyret[1]"]) if "dlyret[1]" in params.index else float(params.iloc[1]),
        omega=float(params["omega"]) / (100.0 ** 2),
        alpha=float(params["alpha[1]"]),
        gamma=float(params["gamma[1]"]),
        beta=float(params["beta[1]"]),
        nu=float(params["nu"]),
        sigma0_sq=sigma0_sq,
        mean0=mean0,
        residuals=standardized.astype(np.float64),
        loglik=float(fit.loglikelihood),
    )


def build_residual_pool(
    fit: GARCHFit,
    df_features: pd.DataFrame,
    confounders: list[str],
    k_neighbours: int = 500,
) -> np.ndarray:
    """Selects feature-conditional standardized residuals via k-NN.

    Restricts the residual draw pool to historical days whose confounder
    state was closest to today's in scaled Euclidean distance. This turns
    the unconditional FHS into a nonparametric conditional bootstrap where
    the conditioning set is the Fama-French + EWMA + analyst feature state.

    Args:
        fit: Fitted GARCH instance with standardized residuals.
        df_features: Full feature matrix aligned with the return series.
        confounders: Column names forming the conditioning state.
        k_neighbours: Number of nearest historical days to retain.

    Returns:
        Length-k array of standardized residuals drawn from similar states.
    """
    x = df_features[confounders].dropna().to_numpy(dtype=np.float64)
    if x.shape[0] <= k_neighbours:
        return fit.residuals

    mu_x = x.mean(axis=0)
    sd_x = x.std(axis=0) + 1e-12
    x_std = (x - mu_x) / sd_x

    knn = NearestNeighbors(n_neighbors=k_neighbours, algorithm="auto").fit(x_std[:-1])
    _, idx = knn.kneighbors(x_std[-1:])
    neighbour_idx = idx.ravel()

    n_resid = fit.residuals.size
    offset = x.shape[0] - n_resid
    aligned_idx = neighbour_idx - offset
    aligned_idx = aligned_idx[(aligned_idx >= 0) & (aligned_idx < n_resid)]

    return fit.residuals[aligned_idx] if aligned_idx.size >= 100 else fit.residuals


def simulate_paths(
    fit: GARCHFit,
    residual_pool: np.ndarray,
    horizon_days: int,
    n_paths: int = 100_000,
    seed: int = 42,
) -> np.ndarray:
    """Simulates terminal log-returns via filtered historical simulation.

    Vectorised across paths: at each time step we draw one standardized
    residual per path in a single np.random call, update the full
    conditional-variance vector with the GJR recursion, and accumulate log
    returns. No Python-level loops over paths — only one loop over time.

    Args:
        fit: Fitted GARCH instance.
        residual_pool: Array of standardized residuals to draw from.
        horizon_days: Number of trading days to project forward.
        n_paths: Number of Monte Carlo paths.
        seed: Deterministic seed.

    Returns:
        Array of terminal simple returns (S_T / S_0 - 1) of length n_paths.
    """
    rng = np.random.default_rng(seed)
    sigma_sq = np.full(n_paths, fit.sigma0_sq, dtype=np.float64)
    r_lag = np.full(n_paths, fit.mean0, dtype=np.float64)
    cum_logret = np.zeros(n_paths, dtype=np.float64)

    alpha = fit.alpha
    gamma = fit.gamma
    beta = fit.beta
    persistence = alpha + 0.5 * gamma + beta
    if persistence >= 0.999:
        scale = 0.999 / persistence
        alpha *= scale
        gamma *= scale
        beta *= scale

    for _ in range(horizon_days):
        z = rng.choice(residual_pool, size=n_paths, replace=True)
        sigma = np.sqrt(sigma_sq)
        eps = sigma * z
        r_t = fit.mu + fit.phi * r_lag + eps

        r_t = np.maximum(r_t, -0.999)
        cum_logret += np.log1p(r_t)

        neg = (eps < 0).astype(np.float64)
        sigma_sq = fit.omega + alpha * eps**2 + gamma * neg * eps**2 + beta * sigma_sq
        sigma_sq = np.maximum(sigma_sq, 1e-12)
        r_lag = r_t

    return np.expm1(cum_logret)


def build_physical_scenario_table(
    terminal_returns: np.ndarray,
    return_edges: Sequence[float] = (-0.10, -0.03, 0.03, 0.10),
    labels: Sequence[str] | None = None,
) -> tuple[pd.DataFrame, float]:
    """Partitions a simulated physical return distribution into scenario buckets.

    Mirrors build_rnd_scenario_table so the P-measure output is directly
    comparable to the Q-measure output on the same row schema.

    Args:
        terminal_returns: Simulated terminal simple returns from simulate_paths.
        return_edges: Strictly increasing interior return thresholds.
        labels: Optional worst-to-best labels of length K + 1.

    Returns:
        Tuple of (scenario DataFrame, unconditional physical mean return in pct).
    """
    edges = np.asarray(return_edges, dtype=np.float64)
    if labels is None:
        labels = (
            "Severe downside",
            "Moderate downside",
            "Muted / in-line",
            "Moderate upside",
            "Severe upside",
        )

    lo_edges = np.concatenate(([-np.inf], edges))
    hi_edges = np.concatenate((edges, [np.inf]))
    n = terminal_returns.size

    rows = []
    for i, (lo, hi, lbl) in enumerate(zip(lo_edges, hi_edges, labels), start=1):
        mask = (terminal_returns >= lo) & (terminal_returns < hi)
        prob = float(mask.sum()) / n
        cmean = float(terminal_returns[mask].mean()) if mask.any() else np.nan
        rng = (
            f"< {hi * 100:+.0f}%" if np.isneginf(lo)
            else f"> {lo * 100:+.0f}%" if np.isposinf(hi)
            else f"{lo * 100:+.0f}% to {hi * 100:+.0f}%"
        )
        rows.append({
            "scenario": i,
            "description": lbl,
            "range": rng,
            "probability": prob,
            "conditional_mean": cmean * 100 if np.isfinite(cmean) else np.nan,
        })

    return pd.DataFrame(rows), float(terminal_returns.mean()) * 100.0


def compare_q_vs_p(rnd_table: pd.DataFrame, p_table: pd.DataFrame) -> pd.DataFrame:
    """Merges the Q and P scenario tables into a single comparison frame.

    Args:
        rnd_table: Option-implied scenario table from build_rnd_scenario_table.
        p_table: Physical scenario table from build_physical_scenario_table.

    Returns:
        Side-by-side DataFrame with Q, P, and edge (Q minus P) columns.
    """
    out = rnd_table[["description", "range"]].copy()
    out["P_Q"] = rnd_table["probability"].mul(100).round(1)
    out["P_P"] = p_table["probability"].mul(100).round(1)
    out["edge_pp"] = (out["P_Q"] - out["P_P"]).round(1)
    return out

In [11]:
from dataclasses import dataclass
from typing import Tuple
import numpy as np
import pandas as pd
import statsmodels.api as sm
from arch import arch_model
import matplotlib.pyplot as plt
from scipy.optimize import minimize

@dataclass(slots=True)
class GFStats:
    aic: float
    bic: float
    llf: float
    rmse: float
    mae: float
    r_mu: float
    r_sd: float
    r_sk: float
    r_ku: float
    acf_r: np.ndarray
    pacf_r: np.ndarray
    acf_r2: np.ndarray
    pacf_r2: np.ndarray

def f_garch(rts: pd.Series, nl: int = 15) -> Tuple[object, GFStats]:
    r = rts.dropna().astype(np.float64) * 100.0
    mdl = arch_model(r, mean="AR", lags=1, vol="GARCH", p=1, o=1, q=1, dist="studentst", rescale=False)
    f = mdl.fit(disp="off", show_warning=False)
    cv = f.conditional_volatility.to_numpy()
    rs = f.resid.to_numpy()
    v = cv > 0
    std = np.zeros_like(rs)
    std[v] = rs[v] / cv[v]
    cln = std[~np.isnan(std)]
    sq = cln**2
    rmse = float(np.sqrt(np.nanmean(rs**2)))
    mae = float(np.nanmean(np.abs(rs)))
    m1 = float(np.mean(cln))
    dv = cln - m1
    m2 = float(np.mean(dv**2))
    sd = np.sqrt(m2) if m2 > 0 else np.nan
    m3 = float(np.mean(dv**3) / sd**3) if np.isfinite(sd) and sd > 0 else np.nan
    m4 = float(np.mean(dv**4) / sd**4) if np.isfinite(sd) and sd > 0 else np.nan
    a1 = sm.tsa.stattools.acf(cln, nlags=nl, fft=True)[1:]
    p1 = sm.tsa.stattools.pacf(cln, nlags=nl, method="ywm")[1:]
    a2 = sm.tsa.stattools.acf(sq, nlags=nl, fft=True)[1:]
    p2 = sm.tsa.stattools.pacf(sq, nlags=nl, method="ywm")[1:]
    return f, GFStats(float(f.aic), float(f.bic), float(f.loglikelihood), rmse, mae, m1, sd, m3, m4, a1, p1, a2, p2)

def fmt_g_sts(st: GFStats) -> pd.DataFrame:
    return pd.DataFrame([
        {"Stat": "LLF", "Val": f"{st.llf:.2f}"},
        {"Stat": "AIC", "Val": f"{st.aic:.2f}"},
        {"Stat": "BIC", "Val": f"{st.bic:.2f}"},
        {"Stat": "RMSE", "Val": f"{st.rmse:.4f}%"},
        {"Stat": "MAE", "Val": f"{st.mae:.4f}%"},
        {"Stat": "Mu(z)", "Val": f"{st.r_mu:.4f}"},
        {"Stat": "Sd(z)", "Val": f"{st.r_sd:.4f}"},
        {"Stat": "Sk(z)", "Val": f"{st.r_sk:.4f}"},
        {"Stat": "Ku(z)", "Val": f"{st.r_ku:.4f}"}
    ])

def p_g_diag(st: GFStats, out: str = "g_diag.png") -> None:
    fg, ax = plt.subplots(2, 2, figsize=(9.5, 6.5))
    lg = np.arange(1, len(st.acf_r) + 1)
    ci = 0.05
    cfgs = [
        (ax[0, 0], lg, st.acf_r, "ACF(z)"),
        (ax[0, 1], lg, st.pacf_r, "PACF(z)"),
        (ax[1, 0], lg, st.acf_r2, "ACF(z^2)"),
        (ax[1, 1], lg, st.pacf_r2, "PACF(z^2)")
    ]
    for a, x, y, t in cfgs:
        a.bar(x, y, width=0.4, color="#225BE1", zorder=3)
        a.axhline(0, color="black", lw=0.8, zorder=2)
        a.axhspan(-ci, ci, color="gray", alpha=0.2, zorder=1)
        a.set_title(t, fontsize=10, color="#0E1631")
        a.set_ylim(-0.2, 0.2)
        a.grid(False)
    fg.tight_layout()
    fg.savefig(out, dpi=1200, bbox_inches="tight", facecolor="white")
    plt.close(fg)

@dataclass(slots=True)
class HParams:
    v0: float
    kp: float
    th: float
    rh: float
    sg: float

class HPricer:
    def __init__(self, n_q: int = 64):
        r, w = np.polynomial.legendre.leggauss(n_q)
        self.u = 0.5 * (r + 1.0) * 100.0
        self.w = 0.5 * w * 100.0

    def _cf(self, u: np.ndarray, t: float, p: HParams) -> np.ndarray:
        z = u - 0.5j
        a = -z**2 / 2.0 - 1j * z / 2.0
        b = p.kp - p.rh * p.sg * 1j * z
        g = p.sg**2 / 2.0
        d = np.sqrt(b**2 - 4.0 * a * g)
        rm = (b - d) / (2.0 * g)
        rp = (b + d) / (2.0 * g)
        gr = rm / rp
        et = np.exp(-d * t)
        ct = p.kp * (rm * t - (1.0 / g) * np.log((1.0 - gr * et) / (1.0 - gr)))
        dt = rm * (1.0 - et) / (1.0 - gr * et)
        return np.exp(ct * p.th + dt * p.v0)

    def p_c(self, f: float, k: np.ndarray, t: float, d: float, p: HParams) -> np.ndarray:
        k = np.asarray(k, dtype=np.float64)
        phi = self._cf(self.u, t, p)
        lm = np.log(k / f)[:, None]
        ig = np.real(np.exp(-1j * self.u * lm) * phi / (self.u**2 + 0.25))
        iv = np.sum(ig * self.w, axis=1)
        return np.maximum(f * d - k * d * iv / np.pi, 0.0)

    def p_p(self, f: float, k: np.ndarray, t: float, d: float, p: HParams) -> np.ndarray:
        return np.maximum(self.p_c(f, k, t, d, p) + d * (k - f), 0.0)

    def cal(self, f: float, k: np.ndarray, tc: np.ndarray, t: float, d: float) -> HParams:
        def _l(x: np.ndarray) -> float:
            tp = HParams(*x)
            return float(np.sum((self.p_c(f, k, t, d, tp) - tc)**2))
        b = ((1e-4, 1.0), (1e-4, 10.0), (1e-4, 1.0), (-0.99, 0.99), (1e-4, 5.0))
        x0 = np.array([0.05, 2.0, 0.05, -0.1, 0.5])
        r = minimize(_l, x0, method="L-BFGS-B", bounds=b)
        return HParams(*r.x)

def g_h_px(bl: object, sr: Tuple[float, float] | None = None, n: int = 15) -> pd.DataFrame:
    pr = HPricer()
    f, t, d = bl.forward, bl.T, bl.discount
    mp = pr.cal(f, bl.strike_grid, bl.call_curve, t, d)
    es = np.linspace(sr[0], sr[1], n) if sr else np.linspace(f * 0.8, f * 1.2, n)
    return pd.DataFrame({"Strike": es, "Call": pr.p_c(f, es, t, d, mp), "Put": pr.p_p(f, es, t, d, mp)}).round(4)

# fPCA

In [12]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd


class DataLoader:
    """Parses the combined forwards + options workbook into clean frames.

    Attributes:
        path: Workbook location.
        asof: Snapshot date used to compute option time-to-expiry.
        iv_floor: Lower plausibility clip on implied volatility.
        iv_cap: Upper plausibility clip on implied volatility.
    """

    _TICKER_RE = re.compile(
        r"^\s*(?P<root>\S+)\s+\S+\s+"
        r"(?P<expiry>\d{2}/\d{2}/\d{2})\s+"
        r"(?P<type>[CP])(?P<strike>\d+(?:\.\d+)?)"
    )

    def __init__(
        self,
        path: str | Path,
        asof: str | pd.Timestamp = "2026-04-07",
        iv_floor: float = 0.02,
        iv_cap: float = 3.0,
    ):
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"Workbook not found: {self.path}")
        self.asof = pd.Timestamp(asof)
        self.iv_floor = iv_floor
        self.iv_cap = iv_cap

    def load(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Parses all three target sheets in a single workbook open.

        Returns:
            Tuple of (brent_curve, jkm_curve, options_chain).
        """
        with pd.ExcelFile(self.path, engine="openpyxl") as xl:
            brent = self._curve(xl, "Sheet6", "CO")
            jkm = self._curve(xl, "Sheet7", "JKL")
            opt = self._options(xl, "Sheet8")
        return brent, jkm, opt

    def _curve(self, xl: pd.ExcelFile, sheet: str, root: str) -> pd.DataFrame:
        """Parses a Bloomberg forward curve sheet into a wide tenor matrix.

        Args:
            xl: Open ExcelFile handle.
            sheet: Sheet name to parse.
            root: Contract root (e.g. "CO", "JKL").

        Returns:
            DataFrame indexed by business date with integer tenor columns.
        """
        df = xl.parse(sheet, header=0)
        df = df.rename(columns={df.columns[0]: "date"})
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).set_index("date").sort_index()

        tenor_re = re.compile(rf"^{root}(\d+)\s+(?:COMB\s+)?Comdty$", re.IGNORECASE)
        rename = {c: int(m.group(1)) for c in df.columns if (m := tenor_re.match(str(c)))}

        df = df[list(rename)].rename(columns=rename)
        df = df.reindex(columns=sorted(df.columns)).apply(pd.to_numeric, errors="coerce")
        return df.dropna(how="all").dropna(axis=1, how="all")

    def _options(self, xl: pd.ExcelFile, sheet: str) -> pd.DataFrame:
        """Parses the WDS options chain into a long-format frame.

        Args:
            xl: Open ExcelFile handle.
            sheet: Sheet name to parse.

        Returns:
            DataFrame with one row per valid option quote.
        """
        df = xl.parse(
            sheet,
            header=None,
            skiprows=1,
            names=["ticker", "strike_bbg", "type_bbg", "px_last", "px_settle", "ivol"],
        )
        df = df.dropna(subset=["ticker"]).reset_index(drop=True)

        parsed = df["ticker"].astype(str).str.extract(self._TICKER_RE)
        df["root"] = parsed["root"]
        df["expiry"] = pd.to_datetime(parsed["expiry"], format="%m/%d/%y", errors="coerce")
        df["type"] = parsed["type"].map({"C": "Call", "P": "Put"})
        df["strike"] = pd.to_numeric(parsed["strike"], errors="coerce")
        df = df.dropna(subset=["expiry", "type", "strike"]).reset_index(drop=True)

        for c in ("px_last", "px_settle", "ivol"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df["price"] = df["px_settle"].where(df["px_settle"].notna(), df["px_last"])
        df["iv"] = (df["ivol"] / 100.0).clip(lower=self.iv_floor, upper=self.iv_cap)
        df.loc[df["ivol"].isna(), "iv"] = np.nan
        df["T"] = (df["expiry"] - self.asof).dt.days / 365.25

        df = df[(df["T"] > 0) & (df["price"] > 0) & (df["strike"] > 0)].dropna(subset=["price", "strike"])
        df = (
            df.sort_values(["expiry", "type", "strike"])
            .drop_duplicates(subset=["expiry", "type", "strike"], keep="last")
            .reset_index(drop=True)
        )
        return df[["root", "expiry", "T", "type", "strike", "price", "iv", "px_last", "px_settle", "ivol", "ticker"]]

In [13]:
candidates = [
    Path("Sus_Data.xlsx"),
    Path("data/Sus_Data.xlsx"),
    Path("../data/Sus_Data.xlsx"),
]
workbook = next((p for p in candidates if p.exists()), None)
if workbook is None:
    raise FileNotFoundError("Workbook not found in expected locations.")

loader = DataLoader(workbook, asof="2026-04-07")
df_brent, df_jkm, df_opt = loader.load()

In [14]:
df_brent

,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,51.09,51.16,51.16,51.08,50.94,50.80,50.66,50.50,50.36,50.23,...,49.78,49.72,49.69,49.64,49.58,49.51,49.44,49.37,49.30,49.27
2021-01-05,53.60,53.54,53.43,53.24,53.00,52.77,52.54,52.31,52.09,51.87,...,51.09,50.97,50.89,50.80,50.69,50.58,50.47,50.35,50.24,50.17
2021-01-06,54.30,54.18,54.00,53.75,53.45,53.16,52.86,52.56,52.28,52.02,...,51.04,50.89,50.77,50.65,50.52,50.38,50.24,50.12,49.99,49.90
2021-01-07,54.38,54.30,54.13,53.89,53.59,53.30,53.01,52.72,52.44,52.20,...,51.30,51.17,51.06,50.95,50.83,50.70,50.57,50.45,50.33,50.24
2021-01-08,55.99,55.83,55.58,55.27,54.91,54.57,54.24,53.92,53.61,53.33,...,52.27,52.10,51.97,51.83,51.69,51.54,51.39,51.24,51.10,51.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-02,109.03,99.44,91.43,85.95,82.54,80.47,79.02,77.79,76.80,76.06,...,74.06,73.82,73.58,73.37,73.15,72.90,72.67,72.52,72.39,72.29
2026-04-06,109.77,100.12,92.24,86.96,83.73,81.83,80.49,79.30,78.32,77.55,...,75.30,75.03,74.77,74.54,74.30,74.02,73.77,73.60,73.46,73.33
2026-04-07,109.27,100.06,92.77,87.95,84.94,83.13,81.80,80.61,79.60,78.79,...,76.34,76.05,75.78,75.53,75.27,74.98,74.72,74.53,74.37,74.22


In [15]:
df_jkm

,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,14.930,10.325,7.19,6.565,6.215,6.240,6.325,6.490,6.675,7.095,...,6.085,5.820,5.795,5.755,5.860,5.965,5.990,6.355,6.955,6.855
2021-01-05,15.105,9.800,6.85,6.190,6.040,6.075,6.150,6.275,6.550,6.970,...,6.005,5.750,5.720,5.605,5.710,5.810,5.835,6.195,6.780,6.825
2021-01-06,15.550,9.550,6.50,6.065,5.950,5.985,6.065,6.215,6.500,6.925,...,5.980,5.725,5.695,5.700,5.805,5.905,5.930,6.295,6.890,6.810
2021-01-07,17.250,10.775,7.00,6.500,6.240,6.265,6.320,6.465,6.725,7.140,...,6.005,5.750,5.720,5.735,5.840,5.945,5.970,6.335,6.935,6.865
2021-01-08,17.250,12.000,7.29,6.615,6.415,6.440,6.500,6.650,6.900,7.240,...,6.055,5.795,5.770,5.795,5.900,6.005,6.030,6.400,7.005,6.940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-02,19.965,18.340,18.61,18.405,18.205,17.545,17.450,17.770,17.515,17.350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-03,19.965,18.340,18.61,18.405,18.205,17.545,17.450,17.770,17.515,17.350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-06,19.965,18.340,18.61,18.405,18.205,17.545,17.450,17.770,17.515,17.350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
class EmpiricalFPCA:
    """Functional PCA for discretely-sampled forward curves.

    Fits a covariance operator on PCHIP-smoothed curves and extracts the
    leading eigenfunctions via a symmetric eigendecomposition.

    Attributes:
        n_components: Number of principal components to retain.
        grid: Uniform tenor grid produced during fit.
        mean: Empirical mean curve on the grid.
        eigfn: Matrix of eigenfunctions (rows) on the grid.
        eigval: Eigenvalues in descending order.
        evr: Explained variance ratio per retained component.
    """

    def __init__(self, n_components: int = 3):
        if n_components < 1:
            raise ValueError("n_components must be >= 1.")
        self.n_components = n_components
        self.grid: np.ndarray | None = None
        self.mean: np.ndarray | None = None
        self.eigfn: np.ndarray | None = None
        self.eigval: np.ndarray | None = None
        self.evr: np.ndarray | None = None
        self._fit = False

    def _smooth(self, X: pd.DataFrame) -> pd.DataFrame:
        """Projects raw tenor observations onto a uniform integer grid.

        Args:
            X: Raw forward curves with numeric maturity columns.

        Returns:
            DataFrame of PCHIP-interpolated curves on the uniform grid.

        Raises:
            ValueError: If the input is empty or has non-numeric columns.
        """
        if X.empty:
            raise ValueError("Input matrix is empty.")
        try:
            t = np.asarray(X.columns, dtype=float)
        except ValueError as exc:
            raise ValueError("Columns must be numeric maturities.") from exc

        grid = np.arange(int(np.floor(t.min())), int(np.ceil(t.max())) + 1, dtype=float)
        wide = X.copy()
        wide.columns = t
        wide = wide.reindex(columns=np.union1d(t, grid))
        return wide.interpolate(method="pchip", axis=1, limit_direction="both")[grid]

    def fit(self, X: pd.DataFrame) -> "EmpiricalFPCA":
        """Fits the empirical covariance operator and extracts eigenfunctions.

        Args:
            X: Historical training surface with numeric tenor columns.

        Returns:
            Fitted instance.

        Raises:
            ValueError: If n_components exceeds the grid dimension.
            RuntimeError: If no valid curves remain after alignment.
        """
        smooth = self._smooth(X).dropna(how="any", axis=0)
        if smooth.empty:
            raise RuntimeError("Zero valid curves remain after smoothing.")

        Y = smooth.to_numpy()
        n, d = Y.shape
        if self.n_components > d:
            raise ValueError("n_components exceeds grid dimension.")

        self.grid = np.asarray(smooth.columns, dtype=float)
        self.mean = Y.mean(axis=0)
        centered = Y - self.mean
        cov = centered.T @ centered / (n - 1)

        val, vec = np.linalg.eigh(cov)
        self.eigval = val[-self.n_components:][::-1]
        self.eigfn = vec[:, -self.n_components:][:, ::-1].T

        trace = np.trace(cov)
        self.evr = self.eigval / trace if trace > 0 else np.zeros(self.n_components)
        self._fit = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Projects curves onto the fitted eigenfunctions.

        Args:
            X: Curves to decompose.

        Returns:
            DataFrame of principal component scores.

        Raises:
            RuntimeError: If the model has not been fitted.
        """
        if not self._fit:
            raise RuntimeError("Model must be fitted before transform.")
        smooth = self._smooth(X)
        scores = (smooth.to_numpy() - self.mean) @ self.eigfn.T
        return pd.DataFrame(
            scores,
            index=smooth.index,
            columns=[f"PC{i+1}" for i in range(self.n_components)],
        )

In [17]:
fpca_brent = EmpiricalFPCA(n_components=3).fit(df_brent)
fpca_jkm   = EmpiricalFPCA(n_components=3).fit(df_jkm)

In [18]:
fpca_brent.eigval, fpca_brent.evr
fpca_jkm.eigval, fpca_jkm.evr

(array([391511.29812232,   2705.94704726,   1371.5031564 ]),
 array([0.98912401, 0.00683637, 0.003465  ]))

In [19]:
brent_scores = fpca_brent.transform(df_brent)
jkm_scores = fpca_jkm.transform(df_jkm)

In [20]:
brent_scores

,PC1,PC2,PC3
date,,,
2021-01-04,-118.844004,-10.775542,3.102677
2021-01-05,-110.955401,-11.700128,3.421948
2021-01-06,-110.178557,-13.005062,3.770882
2021-01-07,-109.161895,-12.470533,3.561335
2021-01-08,-103.722556,-12.710558,3.622724
...,...,...,...
2026-04-02,25.328602,-26.619283,-14.563678
2026-04-06,31.005664,-25.114976,-13.969002
2026-04-07,35.422878,-23.033612,-12.664179


In [21]:
jkm_scores

,PC1,PC2,PC3
date,,,
2021-01-04,112.139670,-20.033520,40.241481
2021-01-05,111.868479,-20.271756,40.939924
2021-01-06,112.004531,-20.077689,41.202467
2021-01-07,112.120952,-20.157341,39.528542
2021-01-08,112.278998,-20.131392,38.688474
...,...,...,...
2026-04-02,1512.783054,121.274000,-1.776856
2026-04-03,1512.783054,121.274000,-1.776856
2026-04-06,1512.783054,121.274000,-1.776856


In [22]:
print(fpca_brent.eigfn.shape)
print(fpca_brent.eigfn)

print(fpca_jkm.eigfn.shape)
print(fpca_jkm.eigfn)

(3, 24)
[[ 0.27734017  0.26574768  0.25513935  0.24561449  0.2371162   0.22949063
   0.22269702  0.21652808  0.21088102  0.20578563  0.20118328  0.19691726
   0.19284205  0.18873814  0.18468351  0.18074071  0.17686831  0.17308308
   0.16934446  0.16565771  0.1620648   0.15858825  0.15529435  0.15206213]
 [-0.51066672 -0.39173254 -0.28682863 -0.20052671 -0.13227403 -0.07778698
  -0.03446097  0.00176248  0.03271696  0.05914616  0.08062266  0.09891047
   0.11460089  0.12958528  0.14385855  0.15676314  0.16850321  0.17972591
   0.19070086  0.20199156  0.2127032   0.22240921  0.23061523  0.23820984]
 [-0.51310881 -0.19623134  0.01620898  0.14947892  0.21761957  0.24350734
   0.24277751  0.2298718   0.21181416  0.18878394  0.1603465   0.12993382
   0.09681458  0.06096366  0.02526869 -0.01189248 -0.04946495 -0.0866603
  -0.12492189 -0.16242342 -0.20013785 -0.23790142 -0.2742791  -0.31044641]]
(3, 24)
[[ 0.00268935  0.0026914   0.00283981  0.00288661  0.00286482  0.0027757
   0.00272102  0.002

In [23]:
display(pd.DataFrame(fpca_brent.eigfn, columns=fpca_brent.grid, index=["PC1","PC2","PC3"]))
display(pd.DataFrame(fpca_jkm.eigfn, columns=fpca_jkm.grid, index=["PC1","PC2","PC3"]))

,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0
PC1,0.277340,0.265748,0.255139,0.245614,0.237116,0.229491,0.222697,0.216528,0.210881,0.205786,...,0.184684,0.180741,0.176868,0.173083,0.169344,0.165658,0.162065,0.158588,0.155294,0.152062
PC2,-0.510667,-0.391733,-0.286829,-0.200527,-0.132274,-0.077787,-0.034461,0.001762,0.032717,0.059146,...,0.143859,0.156763,0.168503,0.179726,0.190701,0.201992,0.212703,0.222409,0.230615,0.238210
PC3,-0.513109,-0.196231,0.016209,0.149479,0.217620,0.243507,0.242778,0.229872,0.211814,0.188784,...,0.025269,-0.011892,-0.049465,-0.086660,-0.124922,-0.162423,-0.200138,-0.237901,-0.274279,-0.310446


,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0
PC1,0.002689,0.002691,0.002840,0.002887,0.002865,0.002776,0.002721,0.002688,0.002594,0.002500,...,0.004229,0.012672,0.029814,0.059200,0.104406,0.169120,0.256997,0.371703,0.516859,0.696050
PC2,-0.020267,-0.021292,-0.022095,-0.022494,-0.022311,-0.021110,-0.020367,-0.019711,-0.019515,-0.019461,...,0.027827,0.104968,0.207469,0.315578,0.405379,0.447639,0.412005,0.265776,-0.022583,-0.483661
PC3,-0.263769,-0.271457,-0.282405,-0.283865,-0.277243,-0.266240,-0.260525,-0.257036,-0.255029,-0.254114,...,-0.180639,-0.102721,-0.066247,-0.056293,-0.043640,-0.026238,0.001701,0.019703,0.023055,0.007902


# Causal Estimation

In [24]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import MultiLabelBinarizer

class FeatureMatrix:
    """Engineers EWMA confounders and forward cumulative abnormal returns.

    Attributes:
        horizon: Trading days for the forward CAR target.
        span: EWMA span for volatility and momentum confounders.
    """

    _BASE = ("mktrf", "smb", "hml", "rmw", "cma", "rf", "vol_ewm", "mom_ewm")
    _OPT = ("meanest_fy1", "meanest_fy2", "highest_fy1", "highest_fy2",
            "lowest_fy1", "lowest_fy2", "at", "lt", "dlyvol")

    def __init__(self, horizon: int = 5, span: int = 20):
        self.horizon = horizon
        self.span = span

    def transform(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, str, List[str]]:
        """Builds the confounded forward-return feature frame.

        Args:
            df: Base dataset with daily returns and Fama-French factors.

        Returns:
            Tuple of (processed frame, target column name, confounder list).

        Raises:
            ValueError: If the input frame is empty.
            KeyError: If required baseline columns are missing.
        """
        if df.empty:
            raise ValueError("Input DataFrame is empty.")
        required = ["dlyret", "mktrf", "smb", "hml"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"Missing required columns: {missing}")

        out = df.copy()
        out["ar"] = out["dlyret"] - out["mktrf"]
        target = f"car_fwd_{self.horizon}d"
        shifts = [out["ar"].shift(-i) for i in range(1, self.horizon + 1)]
        out[target] = np.add.reduce(shifts)

        ewm = out["dlyret"].ewm(span=self.span, adjust=False)
        out["vol_ewm"] = ewm.std() * np.sqrt(252)
        out["mom_ewm"] = ewm.mean()

        confounders = list(self._BASE) + [c for c in self._OPT if c in out.columns]
        out = out.dropna(subset=[target] + confounders).reset_index(drop=True)
        return out, target, confounders

In [25]:
import numpy as np
import pandas as pd
from typing import List, Dict
from scipy.stats import norm
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MultiLabelBinarizer


class ContinuousCausalEstimator:
    """Vectorized Double Machine Learning for continuous ATE estimation.

    Attributes:
        y_col: Continuous outcome column.
        e_col: Event-array column.
        x_cols: Confounder column names.
        cv: Cross-fitting fold count.
        min_obs: Minimum treatments required per event to retain it.
        seed: Deterministic seed.
    """

    def __init__(
        self,
        y_col: str,
        e_col: str,
        x_cols: List[str],
        cv: int = 5,
        min_obs: int = 1,
        seed: int = 42,
    ):
        self.y_col = y_col
        self.e_col = e_col
        self.x_cols = x_cols
        self.cv = cv
        self.min_obs = min_obs
        self.seed = seed
        self._mlb = MultiLabelBinarizer()

    def _treatments(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extracts the binary treatment matrix from the encoded event column.

        Args:
            df: Source frame containing the event column.

        Returns:
            DataFrame of 0/1 treatment indicators, one column per qualifying event.
        """
        events = df[self.e_col].astype(str).str.findall(r"\d+").apply(lambda xs: [int(x) for x in xs])
        t = pd.DataFrame(self._mlb.fit_transform(events), columns=self._mlb.classes_, index=df.index)
        return t.loc[:, t.sum(axis=0) >= self.min_obs]

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Runs the orthogonalized partially-linear DML estimator.

        Args:
            df: Integrated feature matrix with the target and event columns.

        Returns:
            DataFrame of per-event ATE, standard error, t-statistic and p-value,
            sorted by absolute t-statistic.
        """
        d = df.dropna(subset=[self.y_col, self.e_col]).copy()
        x = d[self.x_cols].to_numpy(dtype=np.float64)
        y = d[self.y_col].to_numpy(dtype=np.float64)

        t_df = self._treatments(d)
        if t_df.empty:
            return pd.DataFrame()
        t = t_df.to_numpy(dtype=np.float64)

        kf = KFold(n_splits=self.cv, shuffle=True, random_state=self.seed)
        base = HistGradientBoostingRegressor(random_state=self.seed)
        multi = MultiOutputRegressor(base)

        t_res = t - cross_val_predict(multi, x, t, cv=kf, n_jobs=-1)
        y_res = y - cross_val_predict(base, x, y, cv=kf, n_jobs=-1)

        ss = np.einsum("ij,ij->j", t_res, t_res)
        keep = ss > 0
        t_res, ss = t_res[:, keep], ss[keep]
        ids = t_df.columns[keep]
        n_obs = t[:, keep].sum(axis=0).astype(int)

        ate = np.einsum("ij,i->j", t_res, y_res) / ss
        eps = y_res[:, None] - t_res * ate[None, :]
        var = np.einsum("ij,ij->j", t_res * eps, t_res * eps) / ss**2
        se = np.sqrt(var)
        tstat = ate / se

        return (
            pd.DataFrame({
                "event_id": ids,
                "n_obs": n_obs,
                "ate": ate,
                "se": se,
                "t_stat": tstat,
                "p_val": 2 * norm.sf(np.abs(tstat)),
            })
            .sort_values("t_stat", key=np.abs, ascending=False)
            .reset_index(drop=True)
        )

In [26]:
class ContinuousDistributionEstimator:
    """Gradient-boosted quantile regression for the conditional return CDF.

    Attributes:
        y_col: Continuous outcome column.
        e_col: Event-array column.
        x_cols: Confounder column names.
        quantiles: Quantile levels at which the CDF is evaluated.
        seed: Deterministic seed.
    """

    def __init__(
        self,
        y_col: str,
        e_col: str,
        x_cols: List[str],
        quantiles: List[float] | None = None,
        seed: int = 42,
    ):
        self.y_col = y_col
        self.e_col = e_col
        self.x_cols = x_cols
        self.quantiles = quantiles or [0.05, 0.25, 0.50, 0.75, 0.95]
        self.seed = seed
        self.models: Dict[float, HistGradientBoostingRegressor] = {}
        self._mlb = MultiLabelBinarizer()

    def _design(self, df: pd.DataFrame, fit: bool) -> np.ndarray:
        """Concatenates confounders with multi-label treatment indicators.

        Args:
            df: Frame containing confounders and events.
            fit: Whether to fit the binarizer or only transform.

        Returns:
            Float64 design matrix.
        """
        events = df[self.e_col].astype(str).str.findall(r"\d+").apply(lambda xs: [int(x) for x in xs])
        codes = self._mlb.fit_transform(events) if fit else self._mlb.transform(events)
        return np.hstack([df[self.x_cols].to_numpy(dtype=np.float64), codes.astype(np.float64)])

    def fit(self, df: pd.DataFrame) -> "ContinuousDistributionEstimator":
        """Fits one HGBT quantile regressor per target quantile.

        Args:
            df: Integrated feature matrix.

        Returns:
            Fitted instance.
        """
        d = df.dropna(subset=[self.y_col, self.e_col]).copy()
        x = self._design(d, fit=True)
        y = d[self.y_col].to_numpy(dtype=np.float64)
        for q in self.quantiles:
            m = HistGradientBoostingRegressor(loss="quantile", quantile=q, random_state=self.seed)
            m.fit(x, y)
            self.models[q] = m
        return self

    def predict_distribution(self, df: pd.DataFrame) -> pd.DataFrame:
        """Predicts the conditional return quantiles, enforcing monotonicity.

        Args:
            df: Feature matrix with target events to evaluate.

        Returns:
            DataFrame of predicted quantiles, monotonically sorted across columns.
        """
        x = self._design(df, fit=False)
        out = pd.DataFrame(index=df.index)
        for q, m in self.models.items():
            out[f"q_{q:.2f}"] = m.predict(x)
        out.iloc[:, :] = np.maximum.accumulate(out.to_numpy(), axis=1)
        return out

In [27]:
import numpy as np
import pandas as pd
from typing import List, Dict
from lifelines import WeibullAFTFitter


class SurvivalDataBuilder:
    """Builds right-censored time-to-event durations from event panel data."""

    @staticmethod
    def build(df: pd.DataFrame, target_id: int, event_col: str = "keydeveventtypeid") -> pd.DataFrame:
        """Computes time-to-next-event for each observation row.

        Args:
            df: Panel data with a datetime column and encoded event sequences.
            target_id: Numerical identifier for the terminal event.
            event_col: Column containing the encoded event sequence strings.

        Returns:
            DataFrame augmented with ``duration`` (days) and ``observed`` (bool).
        """
        d = df.copy()
        d["date"] = pd.to_datetime(d["dlycaldt"])
        mask = d[event_col].astype(str).str.contains(rf"\b{target_id}\b", regex=True)
        event_dates = d.loc[mask, "date"].dropna().sort_values().to_numpy()
        dates = d["date"].to_numpy()

        if event_dates.size == 0:
            d["duration"] = (dates.max() - dates).astype("timedelta64[D]").astype(float)
            d["observed"] = False
            return d[d["duration"] > 0].reset_index(drop=True)

        idx = np.searchsorted(event_dates, dates, side="right")
        censored = idx == event_dates.size
        nxt = np.empty_like(dates)
        nxt[~censored] = event_dates[idx[~censored]]
        nxt[censored] = dates.max()

        d["duration"] = (nxt - dates).astype("timedelta64[D]").astype(float)
        d["observed"] = ~censored
        return d[d["duration"] > 0].reset_index(drop=True)


class WeibullTimingEstimator:
    """Weibull AFT survival model for event timing forecasts.

    Attributes:
        features: Covariate columns passed to the regression.
        penalizer: L2 penalty on the log-likelihood.
        model: Underlying lifelines fitter instance.
    """

    def __init__(self, features: List[str], penalizer: float = 0.01):
        from lifelines import WeibullAFTFitter
        self.features = features
        self.penalizer = penalizer
        self.model = WeibullAFTFitter(penalizer=penalizer)

    def fit(self, df: pd.DataFrame) -> "WeibullTimingEstimator":
        """Fits the AFT model on the survival-augmented frame.

        Args:
            df: Frame containing features plus ``duration`` and ``observed``.

        Returns:
            Fitted instance.
        """
        fit_df = df[self.features + ["duration", "observed"]].dropna()
        self.model.fit(fit_df, duration_col="duration", event_col="observed", show_progress=False)
        return self

    def _survival(self, x: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Extracts the fitted survival function as numpy arrays.

        Args:
            x: Single-row feature frame.

        Returns:
            Tuple of (time grid, survival probability grid).
        """
        sf = self.model.predict_survival_function(x[self.features])
        return sf.index.to_numpy(dtype=float), sf.iloc[:, 0].to_numpy(dtype=float)

    def predict_buckets(self, x: pd.DataFrame, buckets: List[int] | None = None) -> Dict[str, float]:
        """Aggregates survival mass into labelled time buckets.

        Args:
            x: Single-row feature frame.
            buckets: Bucket boundaries in days; defaults to [30, 60, 90].

        Returns:
            Dict mapping bucket label to percentage probability mass.
        """
        buckets = buckets or [30, 60, 90]
        idx, val = self._survival(x)
        s = np.interp(np.asarray(buckets, dtype=float), idx, val)
        probs = {f"0-{buckets[0]} Days": 1.0 - s[0]}
        for i in range(len(buckets) - 1):
            probs[f"{buckets[i]}-{buckets[i+1]} Days"] = s[i] - s[i + 1]
        probs[f"{buckets[-1]}+ Days"] = s[-1]
        return {k: round(v * 100, 2) for k, v in probs.items()}

    def predict_daily(self, x: pd.DataFrame, max_days: int = 252) -> pd.DataFrame:
        """Produces the daily survival, cumulative and marginal probability curves.

        Args:
            x: Single-row feature frame.
            max_days: Upper bound of the daily forecast horizon.

        Returns:
            DataFrame with columns day, survival_prob, cumulative_prob, marginal_prob.
        """
        idx, val = self._survival(x)
        days = np.arange(1, max_days + 1, dtype=float)
        s = np.interp(days, idx, val)
        lag = np.insert(s[:-1], 0, 1.0)
        return pd.DataFrame({
            "day": days.astype(int),
            "survival_prob": s,
            "cumulative_prob": 1.0 - s,
            "marginal_prob": lag - s,
        })

In [28]:
import numpy as np
from scipy.optimize import minimize

def cal_dist(tgt: np.ndarray, mu: np.ndarray, d: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Calibrates Gaussian mixture weights and sigmas to match target probabilities.

    Args:
        tgt: Target probability mass per component.
        mu: Mean event days from asof.
        d: Grid days.

    Returns:
        Tuple of optimized weights and sigmas arrays.
    """
    n = len(mu)

    def loss(x: np.ndarray) -> float:
        w, sig = x[:n], x[n:]
        w = w / w.sum()
        z = (d[:, None] - mu) / sig
        pdf = (np.exp(-0.5 * z**2) / (np.sqrt(2.0 * np.pi) * sig) * w).sum(axis=1)

        tot = np.trapezoid(pdf, d)
        if tot > 0:
            pdf /= tot

        p_est = np.zeros(n)
        for i in range(n):
            lo = max(1, int(mu[i] - 2 * sig[i]))
            hi = min(len(d), int(mu[i] + 2 * sig[i]))
            m = (d >= lo) & (d <= hi)
            p_est[i] = np.trapezoid(pdf[m], d[m])

        return float(np.sum((p_est - tgt)**2))

    res = minimize(
        loss,
        x0=np.concatenate([np.ones(n) / n, np.full(n, 7.0)]),
        bounds=[(0.01, 1.0)] * n + [(1.0, 45.0)] * n,
        method="SLSQP"
    )

    opt = res.x
    w, sig = opt[:n], opt[n:]
    return w / w.sum(), sig

In [29]:
import numpy as np
import pandas as pd

def run_calibration() -> pd.DataFrame:
    """Executes SLSQP calibration for the 6-month catalyst timing distribution."""
    asof = pd.Timestamp("2026-04-11")

    anchors = pd.DatetimeIndex([
        "2026-05-12",
        "2026-06-30",
        "2026-08-15"
    ])

    mu = (anchors - asof).days.values.astype(np.float64)
    tgt = np.array([0.45, 0.20, 0.35], dtype=np.float64)
    d = np.arange(1, 181, dtype=np.float64)

    w, sig = cal_dist(tgt, mu, d)

    return pd.DataFrame({
        "anchor_date": anchors,
        "days_to_event": mu.astype(np.int64),
        "target_prob": tgt,
        "calibrated_weight": w,
        "calibrated_sigma": sig
    })

df_params = run_calibration()
print(df_params.to_string(index=False))

anchor_date  days_to_event  target_prob  calibrated_weight  calibrated_sigma
 2026-05-12             31         0.45           0.453434          7.037338
 2026-06-30             80         0.20           0.195840          6.994632
 2026-08-15            126         0.35           0.350726          7.037948


In [30]:
from dataclasses import dataclass
from typing import Sequence

import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid

@dataclass(slots=True, frozen=True)
class TimingAnchor:
    """Gaussian mixture component.

    Args:
        label: Identifier for the anchor.
        date: Mean date of the anchor.
        sigma_days: Standard deviation in days.
        weight: Unnormalized mixture weight.
    """
    label: str
    date: pd.Timestamp
    sigma_days: float
    weight: float

@dataclass(slots=True)
class PolicyTimingDistribution:
    """Truncated horizon mixture of Gaussians with optional uniform tail.

    Args:
        asof: Base date for calculations.
        anchors: Sequence of mixture components.
        tail_start: Start date for the uniform slippage tail.
        tail_end: End date for the uniform slippage tail.
        tail_weight: Unnormalized weight for the uniform tail.
        horizon_days: Maximum number of days to model.
    """
    asof: pd.Timestamp
    anchors: tuple[TimingAnchor, ...]
    tail_start: pd.Timestamp
    tail_end: pd.Timestamp
    tail_weight: float = 0.0
    horizon_days: int = 252

    grid_days: np.ndarray = None
    density: np.ndarray = None

    def __post_init__(self) -> None:
        object.__setattr__(self, 'grid_days', np.arange(1, self.horizon_days + 1, dtype=np.float64))
        object.__setattr__(self, 'density', self._mixture_pdf(self.grid_days))

    def _mixture_pdf(self, d: np.ndarray) -> np.ndarray:
        if not self.anchors:
            return np.zeros_like(d)

        w = np.array([a.weight for a in self.anchors], dtype=np.float64)
        mu = np.array([(a.date - self.asof).days for a in self.anchors], dtype=np.float64)
        sig = np.array([a.sigma_days for a in self.anchors], dtype=np.float64)

        m = (w > 0) & (sig > 0)
        w, mu, sig = w[m], mu[m], sig[m]

        z = (d[:, None] - mu) / sig
        pdf = (np.exp(-0.5 * z**2) / (np.sqrt(2.0 * np.pi) * sig) * w).sum(axis=1)

        if self.tail_weight > 0:
            t0 = (self.tail_start - self.asof).days
            t1 = (self.tail_end - self.asof).days
            span = t1 - t0
            if span > 0:
                mask = (d >= t0) & (d <= t1)
                pdf[mask] += self.tail_weight / span

        tot = np.trapezoid(pdf, d)
        if tot > 0:
            pdf /= tot

        return pdf

    def predict_daily(self, max_days: int | None = None) -> pd.DataFrame:
        """Generates daily probability metrics.

        Args:
            max_days: Optional truncation limit for the daily grid.

        Returns:
            DataFrame containing day index, marginal, cumulative, and survival probabilities.
        """
        n = self.horizon_days if max_days is None else min(max_days, self.horizon_days)
        d = self.grid_days[:n]
        marg = self.density[:n]

        s = marg.sum()
        pmf = marg / s if s > 0 else marg
        cdf = cumulative_trapezoid(marg, d, initial=0.0)

        c = cdf[-1]
        if c > 0:
            cdf /= c

        return pd.DataFrame({
            "day": d.astype(np.int64),
            "marginal_prob": pmf,
            "cumulative_prob": cdf,
            "survival_prob": 1.0 - cdf,
        })

    def predict_buckets(self, anchors_only: bool = False) -> pd.DataFrame:
        """Calculates integrated probability mass for each anchor bucket.

        Args:
            anchors_only: If True, excludes the uniform tail bucket.

        Returns:
            DataFrame containing integrated probabilities per defined period.
        """
        d = self.grid_days
        f = self.density

        labels = [a.label for a in self.anchors]
        mu = np.array([(a.date - self.asof).days for a in self.anchors])
        sig = np.array([a.sigma_days for a in self.anchors])

        lo = np.maximum(1, mu - 2 * sig).astype(int)
        hi = np.minimum(self.horizon_days, mu + 2 * sig).astype(int)

        rows = []
        for l, start_d, end_d in zip(labels, lo, hi):
            mask = (d >= start_d) & (d <= end_d)
            p = float(np.trapezoid(f[mask], d[mask]))
            rows.append({
                "label": l,
                "start": self.asof + pd.Timedelta(days=int(start_d)),
                "end": self.asof + pd.Timedelta(days=int(end_d)),
                "probability": p
            })

        if not anchors_only and self.tail_weight > 0:
            cov = sum(r["probability"] for r in rows)
            rows.append({
                "label": "Slippage and or later announcement",
                "start": self.tail_start,
                "end": self.tail_end,
                "probability": max(0.0, 1.0 - cov)
            })

        return pd.DataFrame(rows)

In [31]:
asof = pd.Timestamp("2026-04-07")

anchors = (
    TimingAnchor("Federal Budget night", pd.Timestamp("2026-05-12"), 3.0, 0.40),
    TimingAnchor("End of financial year papers", pd.Timestamp("2026-06-30"), 7.0, 0.20),
    TimingAnchor("Q3 detailed policy release", pd.Timestamp("2026-08-15"), 14.0, 0.25),
)

timing = PolicyTimingDistribution(
    asof=asof,
    anchors=anchors,
    tail_start=pd.Timestamp("2026-10-01"),
    tail_end=pd.Timestamp("2026-12-31"),
    tail_weight=0.15,
    horizon_days=300,
)

print(timing.predict_buckets().to_string(index=False))
print(f"\nGrid integral check: {np.trapezoid(timing.density, timing.grid_days):.4f}")

                             label      start        end  probability
              Federal Budget night 2026-05-06 2026-05-18     0.380375
      End of financial year papers 2026-06-16 2026-07-14     0.193297
        Q3 detailed policy release 2026-07-18 2026-09-12     0.239233
Slippage and or later announcement 2026-10-01 2026-12-31     0.187096

Grid integral check: 1.0000


In [32]:
matrix_builder = FeatureMatrix(horizon=126, span=20)
df_features, target, confounders = matrix_builder.transform(df_main)

In [33]:
df_features

,dlycaldt,dlyprc,dlyret,dlyvol,curcd,at,lt,ni,revt,meanest_fy1,...,cma,rf,keydeveventtypeid,eventtype,category,headline,ar,car_fwd_126d,vol_ewm,mom_ewm
0,2022-06-06,23.76,0.033044,1651320.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,...,0.0017,0.0000,None,None,None,None,0.029744,0.120435,0.443645,-0.002715
1,2022-06-07,24.11,0.014731,2265246.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,...,0.0065,0.0000,None,None,None,None,0.004731,0.132898,0.344974,-0.001053
2,2022-06-08,25.02,0.037744,1899037.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,...,-0.0025,0.0000,None,None,None,None,0.047944,0.082339,0.393561,0.002642
3,2022-06-09,24.89,-0.005196,1451446.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,...,0.0021,0.0000,None,None,None,None,0.019104,0.027916,0.342604,0.001895
4,2022-06-10,24.12,-0.030936,1077745.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,...,0.0055,0.0000,None,None,None,None,-0.000936,0.034260,0.362597,-0.001232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,2025-06-26,15.65,0.009677,663045.0,USD,61264.0,25111.0,3573.0,13179.0,0.96,...,-0.0003,0.0002,None,None,None,None,0.000877,-0.080231,0.349718,0.001617
767,2025-06-27,15.42,-0.014697,952485.0,USD,61264.0,25111.0,3573.0,13179.0,0.96,...,0.0015,0.0002,None,None,None,None,-0.019297,-0.064225,0.341670,0.000063
768,2025-06-30,15.43,0.000649,924353.0,USD,61264.0,25111.0,3573.0,13179.0,0.96,...,0.0017,0.0002,None,None,None,None,-0.004251,-0.045457,0.325005,0.000119
769,2025-07-01,15.57,0.009073,639689.0,USD,61264.0,25111.0,3573.0,13179.0,0.96,...,0.0153,0.0002,None,None,None,None,0.010073,-0.047087,0.312091,0.000972


In [34]:
causal_estimator = ContinuousCausalEstimator(
    y_col=target,
    e_col='keydeveventtypeid',
    x_cols=confounders
)
ate_results = causal_estimator.fit_transform(df_features)

In [35]:
ate_results

,event_id,n_obs,ate,se,t_stat,p_val
0,156,1,0.103830,0.017336,5.989159,2.109294e-09
1,25,1,0.122221,0.027554,4.435673,9.178519e-06
2,65,13,-0.041290,0.015396,-2.681940,7.319650e-03
3,28,6,0.046241,0.017992,2.570053,1.016830e-02
4,42,1,0.083217,0.034331,2.423985,1.535124e-02
5,48,6,0.080627,0.039889,2.021264,4.325248e-02
6,29,12,0.044676,0.022404,1.994114,4.613954e-02
7,41,2,-0.036851,0.019846,-1.856872,6.332942e-02
8,81,6,-0.061680,0.035482,-1.738337,8.215150e-02
9,16,12,0.030158,0.017886,1.686133,9.177017e-02


In [36]:
dist_estimator = ContinuousDistributionEstimator(
    y_col=target,
    e_col='keydeveventtypeid',
    x_cols=confounders,
    quantiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)
dist_estimator.fit(df_features)

In [37]:
recent_events = df_features.dropna(subset=['keydeveventtypeid']).tail()
event_distributions = dist_estimator.predict_distribution(recent_events)

In [38]:
recent_events

,dlycaldt,dlyprc,dlyret,dlyvol,curcd,at,lt,ni,revt,meanest_fy1,...,cma,rf,keydeveventtypeid,eventtype,category,headline,ar,car_fwd_126d,vol_ewm,mom_ewm
727,2025-04-30,12.95,-0.009181,1095915.0,USD,61264.0,25111.0,3573.0,13179.0,1.03,...,0.0008,0.0002,[np.int64(23)],['Client Announcements'],['Customer/Product Related'],['Woodside Energy Group Ltd Signs Agreement fo...,-0.009481,0.052244,0.453555,0.000112
733,2025-05-08,13.15,0.016229,1060884.0,USD,61264.0,25111.0,3573.0,13179.0,1.03,...,0.0009,0.0002,"[np.int64(16), np.int64(163)]","['Executive/Board Changes - Other', 'Investor ...","['Corporate Structure Related', 'Investor Acti...","[""Woodside Energy Group Ltd Elects Tony O'neil...",0.008829,0.115532,0.396912,0.001556
737,2025-05-14,14.29,0.006338,974868.0,USD,61264.0,25111.0,3573.0,13179.0,1.03,...,0.0025,0.0002,[np.int64(42)],['Debt Financing Related'],['Announced/Completed Transactions'],['Woodside Energy Group Ltd Prices US Bond Off...,0.005538,0.109629,0.404439,0.008164
740,2025-05-19,14.03,-0.003551,580892.0,USD,61264.0,25111.0,3573.0,13179.0,0.98,...,-0.0001,0.0002,[np.int64(149)],['Conferences'],['Results Announcements/Corporate Communicatio...,"['Informa Australia Pty Ltd., Pilbara Summit, ...",-0.003651,0.150199,0.362696,0.004500
764,2025-06-24,15.58,-0.037083,1423483.0,USD,61264.0,25111.0,3573.0,13179.0,0.96,...,0.0031,0.0002,"[np.int64(81), np.int64(81), np.int64(81), np....","['M&A Transaction Closings', 'M&A Transaction ...","['Announced/Completed Transactions', 'Announce...",['Stonepeak Partners LP completed the acquisit...,-0.048783,-0.090228,0.382249,0.001390


In [39]:
event_distributions

,q_0.05,q_0.10,q_0.25,q_0.50,q_0.75,q_0.90,q_0.95
727,-0.146622,-0.015736,0.037704,0.067921,0.099424,0.116449,0.133739
733,-0.145961,-0.055104,0.007262,0.070697,0.108185,0.121134,0.128156
737,-0.156766,-0.039017,0.013343,0.094254,0.110548,0.137412,0.145446
740,-0.180924,-0.057407,-0.013267,0.077005,0.109857,0.149669,0.149848
764,-0.169171,-0.077959,-0.035007,0.050990,0.104390,0.129563,0.137933


In [40]:
df_surv = SurvivalDataBuilder.build(df_features, target_id=83)

timing_est = WeibullTimingEstimator(features=confounders, penalizer=0.01)
timing_est.fit(df_surv)

state_t0 = df_surv.tail(1)

timing_dist = timing_est.predict_buckets(state_t0, buckets=[30, 60, 90])
df_timing = pd.DataFrame(list(timing_dist.items()), columns=['Time Horizon', 'Probability (%)'])

continuous_dist = timing_est.predict_daily(state_t0, max_days=90)
peak_hazards = continuous_dist.sort_values('marginal_prob', ascending=False).head(5)

In [41]:
continuous_dist

,day,survival_prob,cumulative_prob,marginal_prob
0,1,1.000000,2.554762e-07,2.554762e-07
1,2,0.999999,1.321849e-06,1.066373e-06
2,3,0.999997,3.457374e-06,2.135525e-06
3,4,0.999993,6.839315e-06,3.381941e-06
4,5,0.999988,1.160951e-05,4.770200e-06
...,...,...,...,...
85,86,0.990172,9.828327e-03,2.675519e-04
86,87,0.989898,1.010223e-02,2.738990e-04
87,88,0.989624,1.037612e-02,2.738990e-04
88,89,0.989344,1.065640e-02,2.802758e-04


In [42]:
peak_hazards

,day,survival_prob,cumulative_prob,marginal_prob
89,90,0.989059,0.010941,0.000285
88,89,0.989344,0.010656,0.000280
87,88,0.989624,0.010376,0.000274
86,87,0.989898,0.010102,0.000274
85,86,0.990172,0.009828,0.000268


In [43]:
df_timing.to_string(index=False)

'Time Horizon  Probability (%)\n   0-30 Days             0.08\n  30-60 Days             0.34\n  60-90 Days             0.67\n    90+ Days            98.91'

# Options

In [44]:
df_opt

,root,expiry,T,type,strike,price,iv,px_last,px_settle,ivol,ticker
0,2WDS,2026-04-09,0.005476,Call,28.5,3.565,0.745070,3.565,3.565,74.50698,2WDS AU 04/09/26 C28.5 Equity
1,2WDS,2026-04-09,0.005476,Call,29.0,3.065,NaN,3.065,3.065,NaN,2WDS AU 04/09/26 C29 Equity
2,2WDS,2026-04-09,0.005476,Call,29.5,2.565,0.542726,2.565,2.565,54.27263,2WDS AU 04/09/26 C29.5 Equity
3,2WDS,2026-04-09,0.005476,Call,30.0,2.070,NaN,2.070,2.070,NaN,2WDS AU 04/09/26 C30 Equity
4,2WDS,2026-04-09,0.005476,Call,30.5,1.595,0.555944,1.595,1.595,55.59438,2WDS AU 04/09/26 C30.5 Equity
...,...,...,...,...,...,...,...,...,...,...,...
1392,WDS,2028-12-21,2.707734,Put,36.0,8.215,0.321290,8.215,8.215,32.12903,WDS AU 12/21/28 P36 Equity
1393,WDS,2028-12-21,2.707734,Put,37.0,8.845,0.299948,8.845,8.845,29.99479,WDS AU 12/21/28 P37 Equity
1394,WDS,2028-12-21,2.707734,Put,38.0,9.465,0.298211,9.465,9.465,29.82107,WDS AU 12/21/28 P38 Equity
1395,WDS,2028-12-21,2.707734,Put,39.0,10.075,0.295053,10.075,10.075,29.50533,WDS AU 12/21/28 P39 Equity


In [45]:
@dataclass(slots=True)
class SABRParams:
    """Calibrated SABR parameters for a single expiry.

    Attributes:
        alpha: Initial volatility level.
        beta: CEV exponent held fixed in calibration.
        rho: Correlation between forward and vol processes.
        nu: Volatility of volatility.
        rmse: In-sample RMSE against market implied volatilities.
    """

    alpha: float
    beta: float
    rho: float
    nu: float
    rmse: float


class SABR:
    """Hagan 2002 SABR implied-volatility formula with Obłój 2008 stabilization.

    Provides the HKLW lognormal implied volatility and a fast 2D calibration
    that fixes beta and pins alpha to the ATM volatility by solving the
    induced cubic, reducing the optimization to (rho, nu).
    """

    @staticmethod
    def iv(f: float, k: np.ndarray, t: float, alpha: float, beta: float, rho: float, nu: float) -> np.ndarray:
        """Computes SABR lognormal implied volatility on a strike vector.

        Args:
            f: Forward price.
            k: Strike or array of strikes.
            t: Time to expiry in years.
            alpha: SABR alpha.
            beta: SABR beta.
            rho: SABR rho.
            nu: SABR nu.

        Returns:
            Array of lognormal implied volatilities matching the shape of ``k``.
        """
        k = np.asarray(k, dtype=float)
        eps = 1e-12
        one_b = 1.0 - beta
        fk = f * k
        log_fk = np.log(f / k)

        fk_pow = fk ** (one_b / 2.0)
        pre = alpha / (fk_pow * (1.0 + one_b**2 / 24.0 * log_fk**2 + one_b**4 / 1920.0 * log_fk**4))

        z = (nu / alpha) * fk_pow * log_fk
        sqrt_term = np.sqrt(1.0 - 2.0 * rho * z + z**2)
        x_z = np.log((sqrt_term + z - rho) / (1.0 - rho))
        ratio = np.where(np.abs(z) < 1e-07, 1.0, z / np.where(np.abs(x_z) < eps, eps, x_z))

        correction = 1.0 + (
            one_b**2 / 24.0 * alpha**2 / (fk_pow**2)
            + 0.25 * rho * beta * nu * alpha / fk_pow
            + (2.0 - 3.0 * rho**2) / 24.0 * nu**2
        ) * t

        return pre * ratio * correction

    @classmethod
    def calibrate(
        cls,
        f: float,
        k: np.ndarray,
        iv_mkt: np.ndarray,
        t: float,
        beta: float = 1.0,
        weights: np.ndarray | None = None,
    ) -> SABRParams:
        """Calibrates (rho, nu) with alpha pinned to the ATM volatility.

        Fixing beta and solving alpha as the positive real root of the cubic
        induced by the ATM condition reduces the calibration to a
        well-conditioned 2D minimization. Multi-start L-BFGS-B is used to
        avoid local minima.

        Args:
            f: Forward price.
            k: Strike vector of market quotes.
            iv_mkt: Corresponding market implied volatilities.
            t: Time to expiry in years.
            beta: CEV exponent held fixed during calibration.
            weights: Optional per-quote weights; defaults to uniform.

        Returns:
            Fitted SABRParams container including the in-sample RMSE.
        """
        k = np.asarray(k, dtype=float)
        iv_mkt = np.asarray(iv_mkt, dtype=float)
        w = np.ones_like(iv_mkt) if weights is None else np.asarray(weights, dtype=float)

        order = np.argsort(k)
        atm_iv = float(np.interp(f, k[order], iv_mkt[order]))

        def alpha_from_atm(rho: float, nu: float) -> float:
            """Solves the ATM cubic for alpha given (rho, nu, beta, f, t, atm_iv).

            Args:
                rho: Current SABR rho.
                nu: Current SABR nu.

            Returns:
                Smallest positive real root, or the leading-order guess on failure.
            """
            one_b = 1.0 - beta
            f_pow = f ** one_b
            c3 = one_b**2 * t / (24.0 * f_pow**3)
            c2 = 0.25 * rho * beta * nu * t / (f_pow**2)
            c1 = (1.0 + (2.0 - 3.0 * rho**2) / 24.0 * nu**2 * t) / f_pow
            c0 = -atm_iv
            roots = np.roots([c3, c2, c1, c0])
            real = roots[np.abs(roots.imag) < 1e-08].real
            pos = real[real > 0]
            return float(pos.min()) if pos.size else atm_iv * f_pow

        def loss(theta: np.ndarray) -> float:
            """Weighted RMSE between SABR and market IVs for given (rho, nu).

            Args:
                theta: Length-2 array [rho, nu].

            Returns:
                Weighted root-mean-squared error across quotes.
            """
            rho, nu = theta
            alpha = alpha_from_atm(rho, nu)
            model = cls.iv(f, k, t, alpha, beta, rho, nu)
            return float(np.sqrt(np.mean(w * (model - iv_mkt) ** 2)))

        best = None
        for rho0 in (-0.5, -0.2, 0.0, 0.2, 0.5):
            for nu0 in (0.2, 0.5, 1.0, 1.5):
                res = minimize(
                    loss,
                    x0=np.array([rho0, nu0]),
                    method="L-BFGS-B",
                    bounds=[(-0.999, 0.999), (1e-04, 5.0)],
                )
                if res.success and (best is None or res.fun < best.fun):
                    best = res

        rho, nu = best.x
        alpha = alpha_from_atm(rho, nu)
        return SABRParams(alpha=alpha, beta=beta, rho=rho, nu=nu, rmse=float(best.fun))


@dataclass(slots=True)
class BLResult:
    """Container for one expiry's Breeden-Litzenberger output.

    Attributes:
        expiry: Expiry date.
        T: Time to expiry in years.
        forward: Forward price inferred from put-call parity.
        discount: Discount factor.
        rate: Continuously compounded zero rate.
        sabr: Calibrated SABR parameters for this expiry.
        strike_grid: Dense strike grid for the density.
        iv_curve: SABR implied volatilities on the strike grid.
        call_curve: Black-76 call prices on the strike grid.
        density: Risk-neutral density on the strike grid.
        parity_rmse: RMSE of the put-call parity regression.
        arb_violations: Count of negative density entries prior to clipping.
        n_quotes: Quotes retained for this expiry.
    """

    expiry: pd.Timestamp
    T: float
    forward: float
    discount: float
    rate: float
    sabr: SABRParams
    strike_grid: np.ndarray
    iv_curve: np.ndarray
    call_curve: np.ndarray
    density: np.ndarray
    parity_rmse: float
    arb_violations: int
    n_quotes: int

    def moments(self) -> Dict[str, float]:
        """Computes risk-neutral mean, variance, skewness and excess kurtosis.

        Returns:
            Dict of moment estimates via trapezoidal integration.
        """
        k, q = self.strike_grid, self.density
        tot = np.trapezoid(q, k)
        if tot <= 0:
            return {"mean": np.nan, "variance": np.nan, "skew": np.nan, "kurt": np.nan}
        qn = q / tot
        m1 = float(np.trapezoid(k * qn, k))
        dev = k - m1
        m2 = float(np.trapezoid(dev**2 * qn, k))
        sd = np.sqrt(m2) if m2 > 0 else np.nan
        if np.isnan(sd):
            return {"mean": m1, "variance": m2, "skew": np.nan, "kurt": np.nan}
        m3 = float(np.trapezoid(dev**3 * qn, k) / sd**3)
        m4 = float(np.trapezoid(dev**4 * qn, k) / sd**4)
        return {"mean": m1, "variance": m2, "skew": m3, "kurt": m4}

In [46]:

class BreedenLitzenberger:
    """SABR-based risk-neutral density extraction from an option chain.

    For each expiry the pipeline infers the forward via put-call parity,
    synthesizes OTM call prices, inverts Black-76 implied volatilities,
    calibrates a SABR smile with beta fixed, then evaluates the second
    strike derivative of the SABR call surface to obtain the density.

    Attributes:
        roots: Root tickers treated as European.
        min_quotes: Minimum retained quotes required to fit an expiry.
        n_grid: Strike grid resolution.
        tail: Log-moneyness extrapolation on each side of the observed range.
        beta: SABR beta held fixed during calibration.
        min_t: Minimum time-to-expiry required to attempt calibration.
        failures: Populated by :meth:`fit` with per-expiry error messages.
    """

    def __init__(
        self,
        roots: set[str] | None = None,
        min_quotes: int = 10,
        n_grid: int = 400,
        tail: float = 0.5,
        beta: float = 1.0,
        min_t: float = 7.0 / 365.25,
    ):
        self.roots = roots or {"WDSE"}
        self.min_quotes = min_quotes
        self.n_grid = n_grid
        self.tail = tail
        self.beta = beta
        self.min_t = min_t
        self.failures: Dict[pd.Timestamp, str] = {}

    def fit(self, df_opt: pd.DataFrame) -> Dict[pd.Timestamp, BLResult]:
        """Runs the full BL-SABR pipeline over every eligible expiry.

        Args:
            df_opt: Long-format option chain.

        Returns:
            Dict mapping expiry timestamp to BLResult.

        Raises:
            ValueError: If no quotes match the configured roots.
        """
        df = df_opt[df_opt["root"].isin(self.roots)]
        if df.empty:
            raise ValueError(f"No quotes matched {self.roots}.")

        self.failures.clear()
        out: Dict[pd.Timestamp, BLResult] = {}
        for expiry, sub in df.groupby("expiry", sort=True):
            if len(sub) < self.min_quotes:
                self.failures[expiry] = f"Insufficient quotes ({len(sub)} < {self.min_quotes})"
                continue
            try:
                res = self._fit_one(expiry, sub)
                if res is None:
                    self.failures[expiry] = "Pipeline returned None"
                else:
                    out[expiry] = res
            except Exception as exc:
                self.failures[expiry] = f"{type(exc).__name__}: {exc}"
        return out

    def summary(self, results: Dict[pd.Timestamp, BLResult]) -> pd.DataFrame:
        """Flattens a results dict into a per-expiry diagnostic table.

        Args:
            results: Output of :meth:`fit`.

        Returns:
            DataFrame of expiry-level metrics sorted by expiry.
        """
        rows = []
        for expiry, r in results.items():
            m = r.moments()
            rows.append({
                "expiry": expiry,
                "T": r.T,
                "forward": r.forward,
                "rate": r.rate,
                "n_quotes": r.n_quotes,
                "parity_rmse": r.parity_rmse,
                "sabr_alpha": r.sabr.alpha,
                "sabr_rho": r.sabr.rho,
                "sabr_nu": r.sabr.nu,
                "sabr_rmse": r.sabr.rmse,
                "arb_violations": r.arb_violations,
                "rnd_mean": m["mean"],
                "rnd_stdev": np.sqrt(m["variance"]) if np.isfinite(m["variance"]) else np.nan,
                "rnd_skew": m["skew"],
                "rnd_kurt": m["kurt"],
            })
        if not rows:
            return pd.DataFrame(columns=["expiry", "T", "forward", "rate", "n_quotes", "parity_rmse", "sabr_alpha", "sabr_rho", "sabr_nu", "sabr_rmse", "arb_violations", "rnd_mean", "rnd_stdev", "rnd_skew", "rnd_kurt"])
        return pd.DataFrame(rows).sort_values("expiry").reset_index(drop=True)

    def _fit_one(self, expiry: pd.Timestamp, sub: pd.DataFrame) -> BLResult | None:
        """Fits a single expiry's SABR smile and derives the density.

        Args:
            expiry: Timestamp of the expiry.
            sub: Option quotes filtered to this expiry.

        Returns:
            BLResult on success, or None if any stage has insufficient data.
        """
        t = float(sub["T"].iloc[0])
        if t < self.min_t:
            return None

        f, disc, parity_rmse = self._forward(sub)
        if not (np.isfinite(f) and f > 0 and np.isfinite(disc) and disc > 0):
            return None
        r = -np.log(disc) / t

        strikes, calls = self._otm_calls(sub, f, disc)
        if strikes.size < self.min_quotes:
            return None

        iv = self._implied_vol(calls, strikes, f, t, disc)
        ok = np.isfinite(iv) & (iv > 0.02) & (iv < 3.0)
        strikes, iv = strikes[ok], iv[ok]
        strikes, idx = np.unique(strikes, return_index=True)
        iv = iv[idx]
        if strikes.size < self.min_quotes:
            return None

        params = SABR.calibrate(f, strikes, iv, t, beta=self.beta)

        log_m = np.log(strikes / f)
        k_grid = np.linspace(log_m.min() - self.tail, log_m.max() + self.tail, self.n_grid)
        K = f * np.exp(k_grid)

        iv_grid = SABR.iv(f, K, t, params.alpha, params.beta, params.rho, params.nu)
        iv_grid = np.clip(iv_grid, 0.02, 3.0)
        calls_grid = self._black76(f, K, t, iv_grid, disc)
        density = np.exp(r * t) * np.gradient(np.gradient(calls_grid, K), K)

        arb = int(np.sum(density < 0))
        density = np.clip(density, 0.0, None)
        total = np.trapezoid(density, K)
        if total > 0:
            density /= total

        return BLResult(
            expiry=expiry, T=t, forward=float(f), discount=float(disc), rate=float(r),
            sabr=params, strike_grid=K, iv_curve=iv_grid, call_curve=calls_grid,
            density=density, parity_rmse=parity_rmse, arb_violations=arb, n_quotes=int(strikes.size),
        )

    @staticmethod
    def _forward(sub: pd.DataFrame) -> Tuple[float, float, float]:
        """Regresses put-call parity to infer the forward and discount factor.

        Args:
            sub: Option quotes for a single expiry.

        Returns:
            Tuple of (forward, discount, regression RMSE).
        """
        calls = sub[sub["type"] == "Call"].drop_duplicates("strike", keep="last")
        puts = sub[sub["type"] == "Put"].drop_duplicates("strike", keep="last")
        merged = calls[["strike", "price"]].merge(puts[["strike", "price"]], on="strike", suffixes=("_c", "_p"))
        if len(merged) < 3:
            return np.nan, np.nan, np.nan

        k = merged["strike"].to_numpy(float)
        y = (merged["price_c"] - merged["price_p"]).to_numpy(float)
        a = np.column_stack([np.ones_like(k), k])
        beta, *_ = np.linalg.lstsq(a, y, rcond=None)
        disc = -beta[1]
        if disc <= 0:
            return np.nan, np.nan, np.nan
        f = beta[0] / disc
        rmse = float(np.sqrt(np.mean((y - a @ beta) ** 2)))
        return float(f), float(disc), rmse

    @staticmethod
    def _otm_calls(sub: pd.DataFrame, f: float, disc: float) -> Tuple[np.ndarray, np.ndarray]:
        """Builds an OTM call curve, converting OTM puts via put-call parity.

        Args:
            sub: Option quotes for a single expiry.
            f: Forward price.
            disc: Discount factor.

        Returns:
            Tuple of (sorted strikes, synthetic call prices).
        """
        calls = sub[sub["type"] == "Call"]
        puts = sub[sub["type"] == "Put"]
        kc, vc = calls["strike"].to_numpy(float), calls["price"].to_numpy(float)
        kp, vp = puts["strike"].to_numpy(float), puts["price"].to_numpy(float)

        c_mask = kc >= f
        p_mask = kp < f
        kp_o, vp_o = kp[p_mask], vp[p_mask]
        synthetic = vp_o + disc * (f - kp_o)

        k = np.concatenate([kp_o, kc[c_mask]])
        v = np.concatenate([synthetic, vc[c_mask]])
        valid = v > 0
        k, v = k[valid], v[valid]
        order = np.argsort(k)
        return k[order], v[order]

    @staticmethod
    def _black76(
        f: float | np.ndarray,
        k: float | np.ndarray,
        t: float,
        sigma: float | np.ndarray,
        disc: float,
    ) -> np.ndarray:
        """Black-76 call pricing, vectorized via scipy.special.ndtr.

        Args:
            f: Forward price(s).
            k: Strike(s).
            t: Time to expiry in years.
            sigma: Lognormal volatility value(s).
            disc: Discount factor.

        Returns:
            Call prices broadcast to the largest input shape.
        """
        f = np.asarray(f, float)
        k = np.asarray(k, float)
        sigma = np.asarray(sigma, float)
        vol = np.maximum(sigma * np.sqrt(t), 1e-12)
        d1 = (np.log(f / k) + 0.5 * vol**2) / vol
        return disc * (f * ndtr(d1) - k * ndtr(d1 - vol))

    @classmethod
    def _implied_vol(
        cls,
        price: np.ndarray,
        k: np.ndarray,
        f: float,
        t: float,
        disc: float,
        tol: float = 1e-06,
        max_iter: int = 60,
    ) -> np.ndarray:
        """Vectorized bisection inversion of Black-76 for implied volatility.

        Args:
            price: Market call prices.
            k: Corresponding strikes.
            f: Forward price.
            t: Time to expiry in years.
            disc: Discount factor.
            tol: Absolute tolerance on the bracket width.
            max_iter: Maximum bisection iterations.

        Returns:
            Array of implied vols; NaN where no-arbitrage bounds are violated.
        """
        price = np.asarray(price, float)
        k = np.asarray(k, float)
        lb = disc * np.maximum(f - k, 0.0)
        ub = disc * f
        feasible = (price > lb - 1e-10) & (price < ub + 1e-10)

        lo = np.full_like(price, 1e-04)
        hi = np.full_like(price, 5.0)
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            low = cls._black76(f, k, t, mid, disc) < price
            lo = np.where(low, mid, lo)
            hi = np.where(low, hi, mid)
            if np.max(hi - lo) < tol:
                break
        return np.where(feasible, 0.5 * (lo + hi), np.nan)

In [47]:
all_roots = set(df_opt["root"].unique())
bl = BreedenLitzenberger(
    roots=all_roots,
    tail=0.5,
    min_quotes=5
)
results = bl.fit(df_opt)
summary = bl.summary(results)

In [48]:
results

{Timestamp('2026-04-16 00:00:00'): BLResult(expiry=Timestamp('2026-04-16 00:00:00'), T=0.024640657084188913, forward=32.1052085773575, discount=0.9960657065870485, rate=0.15998165516047766, sabr=SABRParams(alpha=0.43856412625817837, beta=1.0, rho=np.float64(0.300378716533223), nu=np.float64(4.141337393138081), rmse=0.02213279911898948), strike_grid=array([14.86000116, 14.9156142 , 14.97143536, 15.02746544, 15.0837052 ,
        15.14015544, 15.19681694, 15.2536905 , 15.3107769 , 15.36807695,
        15.42559144, 15.48332118, 15.54126697, 15.59942962, 15.65780994,
        15.71640874, 15.77522685, 15.83426509, 15.89352427, 15.95300523,
        16.0127088 , 16.0726358 , 16.13278708, 16.19316347, 16.25376582,
        16.31459497, 16.37565177, 16.43693708, 16.49845174, 16.56019662,
        16.62217258, 16.68438048, 16.74682119, 16.80949558, 16.87240453,
        16.93554891, 16.99892961, 17.06254751, 17.1264035 , 17.19049847,
        17.25483331, 17.31940892, 17.3842262 , 17.44928606, 17.514

In [49]:
summary

,expiry,T,forward,rate,n_quotes,parity_rmse,sabr_alpha,sabr_rho,sabr_nu,sabr_rmse,arb_violations,rnd_mean,rnd_stdev,rnd_skew,rnd_kurt
0,2026-04-16,0.024641,32.105209,0.159982,58,0.022122,0.438564,0.300379,4.141337,0.022133,0,32.103836,2.537891,1.352964,10.098885
1,2026-04-23,0.043806,32.165556,-0.093133,34,0.033982,0.421688,0.169867,2.180891,0.006162,0,32.165886,3.034270,0.689682,5.264798
2,2026-05-21,0.120465,32.216915,0.052697,70,0.024521,0.388256,0.127915,1.257515,0.010478,0,32.211715,4.615118,0.734972,4.950358
3,2026-06-18,0.197125,32.296449,0.036275,87,0.035874,0.363216,0.107354,0.916479,0.010152,0,32.289917,5.506747,0.772651,4.895845
4,2026-07-16,0.273785,32.372629,0.015981,47,0.041225,0.351213,0.020020,0.727668,0.005484,0,32.350354,6.181263,0.641518,4.172974
5,2026-08-20,0.369610,32.441536,0.013153,33,0.022700,0.334326,0.045062,0.715982,0.002125,0,32.389864,6.825006,0.705526,4.192680
6,2026-09-17,0.446270,32.053887,-0.021010,77,0.149932,0.342251,-0.212904,0.720141,0.008659,0,32.015870,7.553331,0.452789,3.866494
7,2026-12-17,0.695414,32.251080,0.009211,72,0.086612,0.305350,-0.004486,0.875592,0.007406,0,31.860514,8.272970,0.554898,4.059199
8,2027-03-18,0.944559,32.003076,0.007318,44,0.202312,0.307045,-0.141550,0.567392,0.008637,0,31.682168,9.306047,0.493422,3.408934
9,2027-06-17,1.193703,32.133011,-0.002061,26,0.080930,0.306559,-0.999000,0.049599,0.002516,0,31.823162,10.266076,0.588559,3.079952


In [50]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import minimize


@dataclass(slots=True)
class HestonParams:
    """Parameters for the Heston Stochastic Volatility model.

    Attributes:
        v0: Initial variance.
        kappa: Rate of mean reversion.
        theta: Long-term variance.
        rho: Correlation between asset returns and variance.
        sigma: Volatility of variance.
    """
    v0: float
    kappa: float
    theta: float
    rho: float
    sigma: float


class HestonFourierPricer:
    """High-performance Fourier-inversion option pricer."""

    def __init__(self, n_quad: int = 64):
        """Initializes the Gauss-Legendre integration grid.

        Args:
            n_quad: Number of integration nodes.
        """
        r, w = np.polynomial.legendre.leggauss(n_quad)
        self._u = 0.5 * (r + 1.0) * 100.0
        self._w = 0.5 * w * 100.0

    def _cf(self, u: np.ndarray, t: float, p: HestonParams) -> np.ndarray:
        """Evaluates the modified Heston characteristic function.

        Args:
            u: Integration grid nodes.
            t: Time to expiration in years.
            p: Heston model parameters.

        Returns:
            Complex array of characteristic function evaluations.
        """
        z = u - 0.5j
        a = -z**2 / 2.0 - 1j * z / 2.0
        b = p.kappa - p.rho * p.sigma * 1j * z
        g = p.sigma**2 / 2.0

        d = np.sqrt(b**2 - 4.0 * a * g)
        rm = (b - d) / (2.0 * g)
        rp = (b + d) / (2.0 * g)
        gr = rm / rp

        et = np.exp(-d * t)
        ct = p.kappa * (rm * t - (1.0 / g) * np.log((1.0 - gr * et) / (1.0 - gr)))
        dt = rm * (1.0 - et) / (1.0 - gr * et)

        return np.exp(ct * p.theta + dt * p.v0)

    def price_c(self, f: float, k: np.ndarray, t: float, d: float, p: HestonParams) -> np.ndarray:
        """Prices a vector of European call options.

        Args:
            f: Forward price.
            k: Array of strike prices.
            t: Time to expiration in years.
            d: Discount factor.
            p: Heston model parameters.

        Returns:
            Array of call option prices.
        """
        k = np.asarray(k, dtype=np.float64)
        phi = self._cf(self._u, t, p)

        lm = np.log(k / f)[:, None]
        ig = np.real(np.exp(-1j * self._u * lm) * phi / (self._u**2 + 0.25))

        iv = np.sum(ig * self._w, axis=1)
        return np.maximum(f * d - k * d * iv / np.pi, 0.0)

    def price_p(self, f: float, k: np.ndarray, t: float, d: float, p: HestonParams) -> np.ndarray:
        """Prices a vector of European put options.

        Args:
            f: Forward price.
            k: Array of strike prices.
            t: Time to expiration in years.
            d: Discount factor.
            p: Heston model parameters.

        Returns:
            Array of put option prices.
        """
        c = self.price_c(f, k, t, d, p)
        return np.maximum(c + d * (k - f), 0.0)

    def calib(self, f: float, k: np.ndarray, tc: np.ndarray, t: float, d: float) -> HestonParams:
        """Calibrates Heston parameters against target prices.

        Args:
            f: Forward price.
            k: Array of strike prices.
            tc: Target call option prices.
            t: Time to expiration in years.
            d: Discount factor.

        Returns:
            Optimized HestonParams object.
        """
        def _loss(x: np.ndarray) -> float:
            tp = HestonParams(v0=x[0], kappa=x[1], theta=x[2], rho=x[3], sigma=x[4])
            ep = self.price_c(f, k, t, d, tp)
            return float(np.sum((ep - tc)**2))

        bnds = ((1e-4, 1.0), (1e-4, 10.0), (1e-4, 1.0), (-0.99, 0.99), (1e-4, 5.0))
        x0 = np.array([0.05, 2.0, 0.05, -0.1, 0.5])

        res = minimize(_loss, x0, method="L-BFGS-B", bounds=bnds)
        return HestonParams(*res.x)


def get_heston_prices(
    bl_result: object,
    strike_range: tuple[float, float] | None = None,
    n_strikes: int = 15
) -> pd.DataFrame:
    """Generates a DataFrame of calibrated Heston prices.

    Args:
        bl_result: Breeden-Litzenberger result.
        strike_range: Tuple of (min_strike, max_strike).
        n_strikes: Number of strikes to evaluate.

    Returns:
        DataFrame containing Strikes, Call Prices, and Put Prices.
    """
    pr = HestonFourierPricer()
    f = bl_result.forward
    t = bl_result.T
    d = bl_result.discount

    mp = pr.calib(f, bl_result.strike_grid, bl_result.call_curve, t, d)

    if strike_range is None:
        es = np.linspace(f * 0.8, f * 1.2, n_strikes)
    else:
        es = np.linspace(strike_range[0], strike_range[1], n_strikes)

    c = pr.price_c(f, es, t, d, mp)
    p = pr.price_p(f, es, t, d, mp)

    return pd.DataFrame({
        "Strike": es,
        "Call": c,
        "Put": p
    }).round(4)

In [51]:
june_expiry = pd.Timestamp("2026-06-18")
if june_expiry in results:
    df_heston_prices = get_heston_prices(results[june_expiry])
    print(df_heston_prices.to_string(index=False))

 Strike   Call    Put
25.8372 9.1202 2.7069
26.7599 7.8840 2.3870
27.6827 6.6226 2.0417
28.6054 5.3565 1.6918
29.5282 4.1543 1.4057
30.4509 3.3544 1.5220
31.3737 2.7941 1.8779
32.2964 2.2527 2.2527
33.2192 1.7422 2.6584
34.1420 1.2323 3.0646
35.0647 0.7398 3.4883
35.9875 0.2512 3.9159
36.9102 0.0000 4.5809
37.8330 0.0000 5.4971
38.7557 0.0000 6.4133


# Figures

In [52]:
def build_rnd_scenario_table(
    bl_result: BLResult,
    return_edges: Sequence[float] = (-0.10, -0.03, 0.03, 0.10),
    labels: Sequence[str] | None = None,
) -> pd.DataFrame:
    r"""Partitions a risk-neutral density into economic return scenarios.

    Converts the RND on the terminal-price grid :math:`S_T` into a density on
    simple returns :math:`r = S_T / F - 1`, then integrates that density across
    analyst-defined return thresholds to produce a scenario table with
    probabilities, return ranges, and mass-weighted expected returns.

    Unlike a quantile-binned table where row probabilities are fixed by
    construction, here the probabilities are data-driven and the return
    ranges are analyst-chosen — the table answers "given what options are
    pricing, how likely is each economic outcome?"

    Args:
        bl_result: Fitted Breeden-Litzenberger result for the target expiry.
        return_edges: Strictly increasing interior return thresholds (not
            including ±infinity). Length ``K`` produces ``K + 1`` buckets.
        labels: Worst-to-best scenario labels of length ``K + 1``. Defaults
            to a 5-bucket severe/moderate/muted/upside/severe partition.

    Returns:
        DataFrame with columns ``scenario``, ``description``, ``range``,
        ``probability``, ``conditional_mean`` (all returns in percent), and
        a final row with the unconditional :math:`\mathbb{E}^{\mathbb{Q}}[r]`.

    Raises:
        ValueError: On non-monotone edges or label-length mismatch.
    """
    edges = np.asarray(return_edges, dtype=float)
    if edges.size < 1 or np.any(np.diff(edges) <= 0):
        raise ValueError("return_edges must be strictly increasing.")
    if labels is None:
        labels = (
            "Severe downside",
            "Moderate downside",
            "Muted / in-line",
            "Moderate upside",
            "Severe upside",
        )
    if len(labels) != edges.size + 1:
        raise ValueError(f"labels length must equal len(return_edges) + 1 ({edges.size + 1}).")

    f = bl_result.forward
    k_grid = bl_result.strike_grid
    q_k = bl_result.density

    # Change of variable: q_r(r) = q_k(F(1+r)) * F.
    r_grid = k_grid / f - 1.0
    q_r = q_k * f
    tot = np.trapezoid(q_r, r_grid)
    if tot > 0:
        q_r = q_r / tot

    lo_edges = np.concatenate(([-np.inf], edges))
    hi_edges = np.concatenate((edges, [np.inf]))

    rows = []
    for i, (lo, hi, lbl) in enumerate(zip(lo_edges, hi_edges, labels), start=1):
        mask = (r_grid >= lo) & (r_grid < hi)
        if not mask.any():
            prob, cmean = 0.0, np.nan
        else:
            rg, qg = r_grid[mask], q_r[mask]
            prob = float(np.trapezoid(qg, rg))
            cmean = float(np.trapezoid(rg * qg, rg) / prob) if prob > 0 else np.nan

        rng = (
            f"< {hi * 100:+.0f}%" if np.isneginf(lo)
            else f"> {lo * 100:+.0f}%" if np.isposinf(hi)
            else f"{lo * 100:+.0f}% to {hi * 100:+.0f}%"
        )
        rows.append({
            "scenario": i,
            "description": lbl,
            "range": rng,
            "probability": prob,
            "conditional_mean": cmean * 100 if np.isfinite(cmean) else np.nan,
        })

    unconditional = float(np.trapezoid(r_grid * q_r, r_grid))
    return pd.DataFrame(rows), unconditional * 100.0


def format_rnd_scenario_table(table: pd.DataFrame, expected_return: float) -> pd.DataFrame:
    """Formats a RND scenario table for presentation.

    Args:
        table: Raw table from ``build_rnd_scenario_table``.
        expected_return: Unconditional expected return in percent.

    Returns:
        Presentation-form DataFrame with percent strings.
    """
    out = table[["description", "range"]].copy()
    out["P()"] = table["probability"].mul(100).round(1).astype(str) + "%"
    out["E[r | scenario]"] = table["conditional_mean"].round(1).astype(str) + "%"
    out.columns = ["Scenario", "Return range", "Probability", "Conditional mean"]
    footer = pd.DataFrame([{
        "Scenario": r"$\mathbb{E}^{\mathbb{Q}}[r]$",
        "Return range": "—",
        "Probability": "100.0%",
        "Conditional mean": f"{expected_return:+.1f}%",
    }])
    return pd.concat([out, footer], ignore_index=True)

In [69]:
from pathlib import Path
from typing import Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter, StrMethodFormatter

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "sans-serif"],
    "mathtext.fontset": "cm",
    "font.size": 10,
    "axes.edgecolor": "#0E1631",
    "axes.linewidth": 0.9,
    "axes.labelcolor": "#0E1631",
    "xtick.color": "#0E1631",
    "ytick.color": "#0E1631",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": True,
    "legend.framealpha": 1.0,
    "legend.edgecolor": "#0E1631",
    "legend.fancybox": False,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 1200,
    "savefig.bbox": "tight",
    "figure.figsize": (10, 6),
})

C = {"d": "#0E1631", "b": "#225BE1", "a": "#4a90c2", "g": "#6B7280"}

def plt_tm(
    d: pd.DataFrame,
    h: int | None = None,
    o: str | Path = "timing_distribution.png",
) -> None:
    """Plots a daily timing distribution as marginal density and cumulative CDF.

    Args:
        d: DataFrame with columns day, marginal_prob, cumulative_prob.
        h: Optional truncation limit for the x-axis in days.
        o: Destination path for the figure.

    Raises:
        KeyError: If required columns are missing.
    """
    req = {"day", "marginal_prob", "cumulative_prob"}
    if m := req - set(d.columns):
        raise KeyError(f"Missing columns: {sorted(m)}")

    df = d if h is None else d.loc[d["day"] <= h]
    x = df["day"].to_numpy()
    m_p = df["marginal_prob"].to_numpy() * 100.0
    c_p = df["cumulative_prob"].to_numpy() * 100.0

    fg, ax = plt.subplots(figsize=(10, 3))
    ax.fill_between(x, m_p, color=C["b"], alpha=0.12)
    l1, = ax.plot(x, m_p, color=C["b"], lw=1.8, label=r"$f(t \mid \mathcal{A})$")
    fmt_ax(ax, r"Calendar days from as-of $t$", r"Marginal density (\%)")

    ax2 = ax.twinx()
    ax2.minorticks_on()
    l2, = ax2.plot(x, c_p, color=C["d"], lw=1.5, ls="--", label=r"$F(t \mid \mathcal{A})$")
    ax2.set_ylabel(r"Cumulative probability (\%)", color=C["d"])
    ax2.tick_params(colors=C["d"])
    ax2.yaxis.set_major_formatter(PercentFormatter(decimals=0))
    ax2.set_ylim(0, 100)

    ax.legend(handles=[l1, l2], loc="center right")
    fg.tight_layout()
    fg.savefig(o)
    plt.close(fg)

def plt_sc(est: object, evt: pd.DataFrame, o: str | Path = "scenario_distribution.png") -> None:
    """Generates a quantile distribution plot of expected catalyst impacts.

    Args:
        est: Fitted ContinuousDistributionEstimator instance.
        evt: DataFrame of recent event observations to evaluate.
        o: Destination path for the figure.
    """
    q = est.predict_distribution(evt).mean(axis=0) * 100.0
    lvl = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
    v = np.array([q[f"q_{l:.2f}"] for l in lvl])

    fg, ax = plt.subplots(figsize=(10, 3))
    ax.plot(v, lvl, color=C["b"], lw=1.8, marker="o", ms=4.5, mfc="white", mec=C["b"], mew=1.2, label=r"$\widehat{F}(r \mid E, \mathbf{x})$")
    ax.axhline(0.5, color=C["g"], lw=0.6, ls=":")
    ax.axvline(0, color=C["g"], lw=0.6, ls=":")
    ax.axvline(v[3], color=C["d"], lw=1.2, ls="--", label=rf"$q_{{0.5}} = {v[3]:+.2f}\%$")

    fmt_ax(ax, r"Forward 5-day CAR $r$ (\%)", r"Quantile level $\tau$")
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:+.0f}"))
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right")
    fg.tight_layout()
    fg.savefig(o)
    plt.close(fg)

def plt_rnd(res: dict, o: str | Path = "implied_density.png") -> None:
    """Generates a SABR-implied risk-neutral density plot.

    Args:
        res: Dictionary mapping expiry timestamps to BLResult objects.
        o: Destination path for the figure.
    """
    if not res:
        return

    _, r = min(res.items(), key=lambda kv: kv[1].T)
    k = r.strike_grid
    qd = r.density
    m = r.moments()
    mu = m["mean"]
    sd = np.sqrt(m["variance"])

    fg, ax = plt.subplots(figsize=(10, 3))
    ax.fill_between(k, qd, color=C["b"], alpha=0.12)
    ax.plot(k, qd, color=C["b"], lw=1.8, label=r"$q(K) = e^{rT}\,\partial^{2}C/\partial K^{2}$")
    ax.axvline(r.forward, color=C["d"], lw=1.2, ls="--", label=rf"$F = {r.forward:.2f}$")
    ax.axvline(mu, color=C["a"], lw=1.2, ls=":", label=rf"$\mathbb{{E}}^{{\mathbb{{Q}}}}[S_T] = {mu:.2f}$")

    fmt_ax(ax, r"Terminal price $S_T$", r"Density $q(K)$")
    ax.set_ylim(0, qd.max() * 1.08)
    ax.legend(loc="upper right", title=rf"$\sigma={sd:.2f},\ \gamma_1={m['skew']:+.2f}$")
    fg.tight_layout()
    fg.savefig(o)
    plt.close(fg)

def plt_fpca(
    f_l: object,
    f_r: object,
    lbl: tuple[str, str] = ("Brent", "JKM"),
    o: str | Path = "fpca_eigenfunctions.png",
) -> None:
    """Plots leading fPCA eigenfunctions for two forward curves.

    Args:
        f_l: Fitted EmpiricalFPCA instance for the left panel.
        f_r: Fitted EmpiricalFPCA instance for the right panel.
        lbl: Panel titles.
        o: Destination path for the figure.

    Raises:
        RuntimeError: If either instance has not been fitted.
    """
    if not f_l._fit or not f_r._fit:
        raise RuntimeError("fPCA instances must be fitted.")

    fg, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
    sty = (("PC1 (level)", C["b"], "-"), ("PC2 (slope)", C["d"], "--"), ("PC3 (curvature)", C["a"], ":"))

    for a, f, n in zip(ax, (f_l, f_r), lbl):
        for i, (l, c, ls) in enumerate(sty):
            e = f.evr[i] * 100.0
            a.plot(f.grid, f.eigfn[i], color=c, ls=ls, lw=1.6, label=rf"{l}, {e:.1f}\%")
        a.axhline(0.0, color=C["g"], lw=0.5, ls=":")
        a.set_title(n, fontsize=10, color=C["d"])
        fmt_ax(a, r"Tenor (months)", "")
        a.legend(loc="best", fontsize=8)

    ax[0].set_ylabel(r"Eigenfunction loading $\phi_k(\tau)$")
    fg.tight_layout()
    fg.savefig(o)
    plt.close(fg)

def plt_12m(df: pd.DataFrame, o: str | Path = "wds_12m.png") -> None:
    """Generates a trailing 12-month historical price chart.

    Args:
        df: DataFrame containing daily calendar dates and prices.
        o: Output filepath for the saved figure.
    """
    m = df["dlycaldt"].max() - pd.DateOffset(months=12)
    f = df[df["dlycaldt"] >= m]
    x = f["dlycaldt"].to_numpy()
    y = f["dlyprc"].to_numpy()

    fg, ax = plt.subplots(figsize=(10, 2))
    ax.fill_between(x, y, color=C["b"], alpha=0.12)
    ax.plot(x, y, color=C["b"], lw=1.8, label="WDS.AX Spot")

    fmt_ax(ax, "Date", "Price (AUD)")
    ax.yaxis.set_major_formatter(StrMethodFormatter("${x:.2f}"))
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(y.min() * 0.95, y.max() * 1.05)
    ax.legend(loc="upper left")

    fg.tight_layout()
    fg.savefig(o)
    plt.close(fg)

plt_12m(df_yf)
plt_fpca(fpca_brent, fpca_jkm)
plt_tm(d=timing.predict_daily(max_days=180))
plt_sc(dist_estimator, recent_events)
plt_rnd(results)

In [54]:
def c_m_p(t_d: PolicyTimingDistribution) -> pd.DataFrame:
    """Calculates integrated probability mass bucketed by calendar month.

    Args:
        t_d: Populated timing distribution object.

    Returns:
        DataFrame of monthly integrated probabilities summing to 1.
    """
    p = t_d.density / np.sum(t_d.density)

    dt_index = t_d.asof + pd.to_timedelta(t_d.grid_days, unit="D")

    df = pd.DataFrame({
        "Month": dt_index.to_period("M").astype(str),
        "Probability": p
    })

    return df.groupby("Month", as_index=False).sum()

df_m_p = c_m_p(timing)
print(df_m_p.to_string(index=False))
print(f"\nSum: {df_m_p['Probability'].sum():.4f}")

  Month  Probability
2026-04 2.348780e-05
2026-05 3.993207e-01
2026-06 1.056664e-01
2026-07 1.314686e-01
2026-08 1.823624e-01
2026-09 2.964809e-02
2026-10 5.112629e-02
2026-11 4.936918e-02
2026-12 5.101481e-02
2027-01 5.463724e-24
2027-02 6.821308e-35

Sum: 1.0000


In [55]:
june_expiry = pd.Timestamp("2026-06-18")
rnd_table, eq_r = build_rnd_scenario_table(
    results[june_expiry],
    return_edges=(-0.10, -0.03, 0.03, 0.10),
)

HORIZON_JUNE = (june_expiry - pd.Timestamp("2026-04-07")).days * (252 / 365.25)
K_NEIGHBOURS = 1000

yf_phys = df_econ_input[["dlycaldt", "dlyret", "dlyvol"]].dropna(subset=["dlyret"]).copy()
ret_ewm = yf_phys["dlyret"].ewm(span=20, adjust=False)
yf_phys["vol_ewm"] = ret_ewm.std() * np.sqrt(252)
garch_confounders = ["dlyvol", "vol_ewm"]

fit = fit_gjr_garch(yf_phys["dlyret"])
pool = build_residual_pool(fit, yf_phys, garch_confounders, k_neighbours=K_NEIGHBOURS)
terminal = simulate_paths(fit, pool, horizon_days=int(round(HORIZON_JUNE)), n_paths=100_000)
p_table, e_p = build_physical_scenario_table(terminal, return_edges=(-0.10, -0.03, 0.03, 0.10))

comparison = compare_q_vs_p(rnd_table, p_table)
print(comparison.to_string(index=False))
print(f"\nE^P[r] = {e_p:+.2f}%    E^Q[r] = {eq_r:+.2f}%    VRP-like wedge = {e_p - eq_r:+.2f}%")

      description       range  P_Q  P_P  edge_pp
  Severe downside      < -10% 27.3 39.5    -12.2
Moderate downside -10% to -3% 17.0  8.2      8.8
  Muted / in-line  -3% to +3% 14.4  7.1      7.3
  Moderate upside +3% to +10% 13.4  7.9      5.5
    Severe upside      > +10% 23.7 37.2    -13.5

E^P[r] = +3.05%    E^Q[r] = -0.02%    VRP-like wedge = +3.07%


In [56]:
import multiprocessing
import numpy as np
import pandas as pd
from itertools import product
from joblib import Parallel, delayed

def crps_1d(x: np.ndarray, y: float) -> float:
    """Computes Continuous Ranked Probability Score efficiently.

    Args:
        x: Predictive sample array.
        y: Realized observation.

    Returns:
        Calculated CRPS value.
    """
    n = x.size
    if n == 0 or not np.isfinite(y):
        return np.nan
    c = 2.0 * np.arange(1, n + 1, dtype=np.float64) - n - 1.0
    return float(np.abs(x - y).mean() - (1.0 / (n * n)) * np.dot(c, np.sort(x)))

def _eval_w(cfg: tuple, w: pd.DataFrame, h: int, n: int) -> dict:
    """Evaluates a single walk-forward configuration.

    Args:
        cfg: Tuple of (origin index, k neighbors, residual scale).
        w: Historical dataframe.
        h: Projection horizon.
        n: Number of paths.

    Returns:
        Dictionary containing configuration and error metrics.
    """
    o, k, s = cfg
    t = w.iloc[: o + 1].copy(deep=False)
    t["vol_ewm"] = t["dlyret"].ewm(span=20, adjust=False).std() * 15.874507866387544

    f = fit_gjr_garch(t["dlyret"])
    p = f.residuals if k is None else build_residual_pool(f, t, ["dlyvol", "vol_ewm"], k)

    sd = o + int(s * 1000) + (0 if k is None else k)
    tm = simulate_paths(f, p * s, h, n, sd)

    r = float(w.loc[o + h, "dlyprc"] / w.loc[o, "dlyprc"] - 1.0)

    return {
        "o": o, "k": k, "s": s,
        "crps": crps_1d(tm, r),
        "mae": abs(float(np.abs(tm).mean()) - abs(r))
    }

def opt_hp(w: pd.DataFrame, h: int, ks: list, ss: list, os: np.ndarray, n: int = 20000, wkr: int | None = None) -> pd.DataFrame:
    """Executes parallelized grid search for hyperparameters.

    Args:
        w: Full historical dataframe.
        h: Evaluation horizon.
        ks: List of k nearest neighbor values to evaluate.
        ss: List of residual scaling factors to evaluate.
        os: Array of origin indices for walk-forward validation.
        n: Number of simulation paths per evaluation.
        wkr: Maximum number of parallel workers.

    Returns:
        Aggregated summary dataframe sorted by CRPS.
    """
    cfgs = list(product(os, ks, ss))
    w_c = wkr or max(1, multiprocessing.cpu_count() - 1)

    rs = Parallel(n_jobs=w_c, backend="loky")(
        delayed(_eval_w)(c, w, h, n) for c in cfgs
    )

    return (
        pd.DataFrame(rs)
        .groupby(["k", "s"], dropna=False)
        .agg(m_crps=("crps", "mean"), m_mae=("mae", "mean"))
        .reset_index()
        .sort_values(["m_crps", "m_mae"])
        .reset_index(drop=True)
    )

In [57]:
h_d = int(round(HORIZON_JUNE))

wf = (
    df_econ_input[["dlycaldt", "dlyprc", "dlyret", "dlyvol"]]
    .dropna(subset=["dlyprc", "dlyret", "dlyvol"])
    .reset_index(drop=True)
    .copy()
 )

# k_g = list(range(100, 2100, 100)) + [None]
# s_g = np.round(np.linspace(0.75, 1.25, 51), 2).tolist()

# l_s = len(wf) - h_d - 1
# o_g = np.arange(500, l_s, 1)
# if o_g.size > 60:
#     o_g = o_g[-60:]

# w_c = max(1, multiprocessing.cpu_count() - 1)

# smr = opt_hp(
#     w=wf,
#     h=h_d,
#     ks=k_g,
#     ss=s_g,
#     os=o_g,
#     n=20000,
#     wkr=w_c
# )

# b = smr.iloc[0]
# b_k = None if pd.isna(b["k"]) else int(b["k"])
# b_s = float(b["s"])

# print(smr.to_string(index=False))
# print(f"\nOptimum: k={b_k}, scale={b_s:.2f}")

In [58]:
import numpy as np
import pandas as pd
from scipy import stats
from typing import Tuple, Callable

def simulate_paths(fit: object, p: np.ndarray, h: int, n: int = 100000, sd: int = 42) -> np.ndarray:
    """Simulates terminal log-returns via filtered historical simulation.

    Args:
        fit: Fitted GARCH instance.
        p: Array of standardized residuals to draw from.
        h: Number of trading days to project forward.
        n: Number of Monte Carlo paths.
        sd: Deterministic seed.

    Returns:
        Array of terminal simple returns of length n.
    """
    g = np.random.default_rng(sd)
    v = np.full(n, fit.sigma0_sq, dtype=np.float64)
    r_l = np.full(n, fit.mean0, dtype=np.float64)
    c_r = np.zeros(n, dtype=np.float64)

    a, gm, b = fit.alpha, fit.gamma, fit.beta
    pr = (a + 0.5 * gm) * float(np.var(p)) + b

    if pr >= 0.999:
        sc = 0.999 / pr
        a *= sc
        gm *= sc
        b *= sc

    v_mx = 6.25 / 252.0

    for _ in range(h):
        z = np.clip(g.choice(p, size=n, replace=True), -5.0, 5.0)
        e = np.sqrt(v) * z
        r_t = np.clip(fit.mu + fit.phi * r_l + e, -0.999, 10.0)
        c_r += np.log1p(r_t)
        v = np.clip(fit.omega + a * e**2 + gm * (e < 0.0) * e**2 + b * v, 1e-12, v_mx)
        r_l = r_t

    return np.expm1(c_r)

def sim_mc(f_g: Callable, w: pd.DataFrame, h: int, k: int | None, s: float, n: int = 300000, sd: int = 20260413) -> np.ndarray:
    """Simulates Monte Carlo paths using GJR-GARCH and conditional residuals.

    Args:
        f_g: GARCH fitting function.
        w: Historical feature dataframe.
        h: Horizon in days.
        k: Number of nearest neighbors.
        s: Residual scaling factor.
        n: Number of paths.
        sd: Random seed.

    Returns:
        Array of terminal returns.
    """
    w_m = w.copy(deep=False)
    w_m["vol_ewm"] = w_m["dlyret"].ewm(span=20, adjust=False).std() * 15.874507866387544
    f = f_g(w_m["dlyret"])
    p = f.residuals if k is None else build_residual_pool(f, w_m, ["dlyvol", "vol_ewm"], k)
    return simulate_paths(f, p * s, h, n, sd)

def mc_sts(tm: np.ndarray) -> pd.DataFrame:
    """Computes statistical profile of terminal returns.

    Args:
        tm: Array of terminal returns.

    Returns:
        DataFrame of statistics.
    """
    p = np.percentile(tm, [1, 5, 25, 50, 75, 95, 99])
    v_95 = float(p[1])
    return pd.DataFrame({
        "metric": [
            "mean", "std", "skew", "kurtosis", "p01", "p05", "p25", "p50", "p75", "p95", "p99",
            "VaR_95", "CVaR_95", "P(r<0)", "P(r>0)", "P(r<-10%)", "P(r>10%)"
        ],
        "value": [
            float(np.mean(tm)),
            float(np.std(tm, ddof=1)),
            float(stats.skew(tm, bias=False)),
            float(stats.kurtosis(tm, bias=False)),
            float(p[0]), v_95, float(p[2]), float(p[3]), float(p[4]), float(p[5]), float(p[6]),
            v_95,
            float(np.mean(tm[tm <= v_95])),
            float(np.mean(tm < 0.0)),
            float(np.mean(tm > 0.0)),
            float(np.mean(tm < -0.10)),
            float(np.mean(tm > 0.10))
        ]
    })

def eval_strad(o: pd.DataFrame, f: float, d: float, tm: np.ndarray, dt: pd.Timestamp) -> Tuple[float, float, float, float, float, float]:
    """Evaluates straddle economics under physical measure.

    Args:
        o: Options dataframe.
        f: Forward price.
        d: Discount factor.
        tm: Terminal returns array.
        dt: Target expiry date.

    Returns:
        Tuple of (Strike, Market Straddle, Expected Payoff, Exp PnL, PV PnL, Win Prob).
    """
    s = o[o["expiry"] == dt]
    c = s[s["type"] == "Call"].set_index("strike")["price"]
    p = s[s["type"] == "Put"].set_index("strike")["price"]

    i = c.index.intersection(p.index)
    x = np.abs(i - f).argmin()
    k = float(i[x])
    sm = float(c.loc[k] + p.loc[k])

    py = np.abs(f * (1.0 + tm) - k)
    ep = float(np.mean(py))

    return k, sm, ep, ep - sm, d * ep - sm, float(np.mean(py > sm))

In [59]:
BEST_K = 1000
BEST_SCALE = 1.0
h = int(round(HORIZON_JUNE))

# Run Simulation
terminal_mc = sim_mc(fit_gjr_garch, wf, h, BEST_K, BEST_SCALE, n=3000000)

# Evaluate Statistics
ret_profile = mc_sts(terminal_mc)
print("Calibrated Monte Carlo returns profile (simple returns):")
print(ret_profile.to_string(index=False))

# Calculate Option Economics
june = results[pd.Timestamp("2026-06-18")]
K, straddle_mkt, e_payoff_phys, exp_pnl_expiry, exp_pnl_pv, prob_profit = eval_strad(
    df_opt, float(june.forward), float(june.discount), terminal_mc, pd.Timestamp("2026-06-18")
)

print("\nStraddle economics under calibrated Monte Carlo P:")
print(f"K={K:.4f}, market premium={straddle_mkt:.4f}, E^P payoff={e_payoff_phys:.4f}")
print(f"Expected PnL expiry={exp_pnl_expiry:+.4f}, PV={exp_pnl_pv:+.4f}, P(profit)={prob_profit:.2%}")

Calibrated Monte Carlo returns profile (simple returns):
   metric     value
     mean  0.030766
      std  0.193393
     skew  1.041646
 kurtosis 18.443613
      p01 -0.441459
      p05 -0.262242
      p25 -0.079451
      p50  0.025464
      p75  0.133262
      p95  0.335233
      p99  0.568683
   VaR_95 -0.262242
  CVaR_95 -0.373024
   P(r<0)  0.433911
   P(r>0)  0.566089
P(r<-10%)  0.212133
 P(r>10%)  0.317532

Straddle economics under calibrated Monte Carlo P:
K=32.5000, market premium=4.2200, E^P payoff=4.5585
Expected PnL expiry=+0.3385, PV=+0.3061, P(profit)=41.61%


In [60]:
from typing import Dict

import jax
import jax.numpy as jnp
from jax.scipy.stats import norm


@jax.jit
def bsm_prc(s: float, k: float, t: float, v: float, r: float, q: float, w: float) -> float:
    """Computes the Black-Scholes-Merton option price.

    Args:
        s: Underlying asset spot price.
        k: Option strike price.
        t: Time to expiration in years.
        v: Implied volatility.
        r: Risk-free interest rate.
        q: Continuous dividend yield.
        w: Option type indicator (1.0 for Call, -1.0 for Put).

    Returns:
        The theoretical option price.
    """
    d1 = (jnp.log(s / k) + (r - q + 0.5 * v ** 2) * t) / (v * jnp.sqrt(t))
    d2 = d1 - v * jnp.sqrt(t)
    return w * s * jnp.exp(-q * t) * norm.cdf(w * d1) - w * k * jnp.exp(-r * t) * norm.cdf(w * d2)


@jax.jit
def calc_grks(s: float, k: float, t: float, v: float, r: float, q: float, w: float) -> Dict[str, float]:
    """Calculates price and first/second-order Greeks using AAD.

    Args:
        s: Underlying asset spot price.
        k: Option strike price.
        t: Time to expiration in years.
        v: Implied volatility.
        r: Risk-free interest rate.
        q: Continuous dividend yield.
        w: Option type indicator (1.0 for Call, -1.0 for Put).

    Returns:
        Dictionary containing the price and respective sensitivities.
    """
    prc = bsm_prc(s, k, t, v, r, q, w)

    g1 = jax.grad(bsm_prc, argnums=(0, 2, 3, 4, 5))(s, k, t, v, r, q, w)
    g2 = jax.hessian(bsm_prc, argnums=(0, 3))(s, k, t, v, r, q, w)

    return {
        "price": prc,
        "delta": g1[0],
        "theta": -g1[1],
        "vega": g1[2],
        "rho": g1[3],
        "epsilon": g1[4],
        "gamma": g2[0][0],
        "vanna": g2[0][1],
        "volga": g2[1][1]
    }


vec_grks = jax.jit(jax.vmap(calc_grks, in_axes=(0, 0, 0, 0, 0, 0, 0)))


def eval_grks_aad(s: jax.Array, k: jax.Array, t: jax.Array, v: jax.Array, r: jax.Array, q: jax.Array, w: jax.Array) -> Dict[str, jax.Array]:
    """Vectorized, XLA-compiled computation of option prices and Greeks via AAD.

    Args:
        s: Array of spot prices.
        k: Array of strike prices.
        t: Array of times to expiration.
        v: Array of implied volatilities.
        r: Array of risk-free rates.
        q: Array of dividend yields.
        w: Array of option type indicators (1.0 for Call, -1.0 for Put).

    Returns:
        Dictionary of arrays containing structural parameters, prices, and Greeks.
    """
    return vec_grks(s, k, t, v, r, q, w)

In [61]:
import jax.numpy as jnp
import numpy as np
import pandas as pd

def append_grks(df: pd.DataFrame, res: dict) -> pd.DataFrame:
    d = df[df["expiry"].isin(res.keys())].dropna(subset=["iv"]).copy()

    f = d["expiry"].map({k: v.forward for k, v in res.items()}).to_numpy(dtype=float)
    r = d["expiry"].map({k: v.rate for k, v in res.items()}).to_numpy(dtype=float)

    t = d["T"].to_numpy(dtype=float)
    k = d["strike"].to_numpy(dtype=float)
    v = d["iv"].to_numpy(dtype=float)
    w = np.where(d["type"] == "Call", 1.0, -1.0)

    s = f * np.exp(-r * t)
    q = np.zeros_like(s)

    g = eval_grks_aad(
        jnp.array(s), jnp.array(k), jnp.array(t),
        jnp.array(v), jnp.array(r), jnp.array(q), jnp.array(w)
    )

    for nk, nv in g.items():
        d[f"bsm_{nk}"] = np.asarray(nv)

    return d

df_opt_grks = append_grks(df_opt, results)

In [62]:
df_opt_grks

,root,expiry,T,type,strike,price,iv,px_last,px_settle,ivol,ticker,bsm_delta,bsm_epsilon,bsm_gamma,bsm_price,bsm_rho,bsm_theta,bsm_vanna,bsm_vega,bsm_volga
67,WDS,2026-04-16,0.024641,Call,18.5,13.575,1.286619,13.575,13.575,128.66190,WDS AU 04/16/26 C18.5 Equity,0.997676,-0.786149,1.125027e-03,13.556381,0.452111,-3.887642,-1.484398e-02,3.647471e-02,2.109058e-01
68,WDS,2026-04-16,0.024641,Call,19.0,13.075,0.447000,13.075,13.075,44.70001,WDS AU 04/16/26 C19 Equity,1.000000,-0.787981,9.975139e-14,13.053648,0.466331,-3.027693,-3.725957e-12,1.123577e-12,1.404869e-10
69,WDS,2026-04-16,0.024641,Call,19.5,12.580,0.432382,12.580,12.580,43.23821,WDS AU 04/16/26 C19.5 Equity,1.000000,-0.787981,2.736408e-13,12.555614,0.478602,-3.107369,-1.004426e-11,2.981445e-12,3.721095e-10
72,WDS,2026-04-16,0.024641,Call,20.5,11.580,0.352694,11.580,11.580,35.26941,WDS AU 04/16/26 C20.5 Equity,1.000000,-0.787981,9.971832e-16,11.559549,0.503146,-3.266721,-4.042117e-14,8.862557e-15,1.649713e-12
73,WDS,2026-04-16,0.024641,Call,21.0,11.080,0.500217,11.080,11.080,50.02169,WDS AU 04/16/26 C21 Equity,1.000000,-0.787981,5.782432e-08,11.061517,0.515418,-3.346404,-1.557824e-06,7.288606e-07,4.258323e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1392,WDS,2028-12-21,2.707734,Put,36.0,8.215,0.321290,8.215,8.215,32.12903,WDS AU 12/21/28 P36 Equity,-0.487885,39.790924,2.504083e-02,8.854029,-63.765285,-0.697540,6.184691e-01,1.976392e+01,-9.310219e-01
1393,WDS,2028-12-21,2.707734,Put,37.0,8.845,0.299948,8.845,8.845,29.99479,WDS AU 12/21/28 P37 Equity,-0.523662,42.708824,2.678775e-02,9.095145,-67.336060,-0.591628,7.341072e-01,1.973825e+01,2.159342e+00
1394,WDS,2028-12-21,2.707734,Put,38.0,9.465,0.298211,9.465,9.465,29.82107,WDS AU 12/21/28 P38 Equity,-0.546533,44.574097,2.680746e-02,9.741839,-70.952423,-0.552857,8.073266e-01,1.963839e+01,4.677874e+00
1395,WDS,2028-12-21,2.707734,Put,39.0,10.075,0.295053,10.075,10.075,29.50533,WDS AU 12/21/28 P39 Equity,-0.570199,46.504276,2.685669e-02,10.378166,-74.605591,-0.504808,8.817265e-01,1.946613e+01,7.729972e+00


In [63]:
import pandas as pd

def extract_atm_straddle(df: pd.DataFrame, exp: str | pd.Timestamp) -> pd.DataFrame:
    """Filters options dataframe for delta-neutral straddle greeks at expiry.

    Args:
        df: DataFrame containing option chain and bsm_* greek columns.
        exp: Target expiration date.

    Returns:
        DataFrame containing Call, Put, and Net Straddle greeks for the ATM strike.
    """
    d = df[df["expiry"] == pd.Timestamp(exp)]
    c = d[d["type"] == "Call"].set_index("strike")
    p = d[d["type"] == "Put"].set_index("strike")

    k = (c["bsm_delta"] + p["bsm_delta"]).abs().idxmin()
    cols = [col for col in df.columns if col.startswith("bsm_")]

    res = pd.DataFrame({
        "Call": c.loc[k, cols],
        "Put": p.loc[k, cols]
    })
    res["Straddle"] = res.sum(axis=1)

    print(f"Delta-Neutral ATM Strike: {k}")
    return res

df_straddle_greeks = extract_atm_straddle(df_opt_grks, "2026-06-18")
display(df_straddle_greeks.round(4))

Delta-Neutral ATM Strike: 32.51


,Call,Put,Straddle
bsm_delta,0.509443,-0.483847,0.025597
bsm_epsilon,-3.220234,3.058435,-0.161798
bsm_gamma,0.088275,0.076619,0.164894
bsm_price,1.702863,2.187821,3.890685
bsm_rho,2.884557,-3.489711,-0.605154
bsm_theta,-5.101344,-4.617977,-9.71932
bsm_vanna,0.147324,0.1328,0.280124
bsm_vega,5.678175,5.67511,11.353285
bsm_volga,-0.049653,-0.076575,-0.126229
